<a href="https://colab.research.google.com/github/DenisBoytsov41/tutors-sentiment-coursework/blob/main/notebooks/03_prepare_labeling_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03_build_target_dataset

Цель ноутбука:
- взять подготовленные артефакты из `02_data_preparation`;
- определить, какие наборы входят в основной целевой контур эксперимента;
- собрать финальный датасет под гипотезу;
- при необходимости подготовить небольшой доменный поднабор на ручную разметку;
- сформировать итоговые `train / validation / test` наборы для следующих этапов.

На этом этапе:
- модель ещё не обучается;
- baseline ещё не запускается;
- финальные метрики ещё не считаются.

Основная логика этапа:
1. загрузить cleaned-артефакты;
2. проверить их состав и структуру;
3. определить основной и вспомогательный контур данных;
4. собрать целевой supervised dataset;
5. подготовить разбиение на `train / validation / test`.

In [388]:
!pip -q install pandas pyarrow matplotlib scikit-learn

In [389]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [390]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

In [391]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## Пути проекта
На этом шаге подключаем папки, которые уже были сформированы на предыдущих этапах.

In [392]:
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/tutors_sentiment_project")

INTERIM_DIR = DRIVE_PROJECT_ROOT / "02_data_interim"
UNIFIED_CORPUS_DIR = INTERIM_DIR / "unified_corpus"
CLEANED_DIR = INTERIM_DIR / "cleaned"
QC_REPORTS_DIR = INTERIM_DIR / "qc_reports"

DATASETS_READY_DIR = DRIVE_PROJECT_ROOT / "04_datasets_ready"
TRAIN_DIR = DATASETS_READY_DIR / "train"
VALIDATION_DIR = DATASETS_READY_DIR / "validation"
TEST_DIR = DATASETS_READY_DIR / "test"
METADATA_DIR = DATASETS_READY_DIR / "metadata"

for folder in [TRAIN_DIR, VALIDATION_DIR, TEST_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("INTERIM_DIR       :", INTERIM_DIR)
print("UNIFIED_CORPUS_DIR:", UNIFIED_CORPUS_DIR)
print("CLEANED_DIR       :", CLEANED_DIR)
print("DATASETS_READY_DIR:", DATASETS_READY_DIR)

INTERIM_DIR       : /content/drive/MyDrive/tutors_sentiment_project/02_data_interim
UNIFIED_CORPUS_DIR: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/unified_corpus
CLEANED_DIR       : /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned
DATASETS_READY_DIR: /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready


## Вспомогательные функции
На этом этапе нам важно посмотреть, какие артефакты сохранились после `02_data_preparation`.

In [393]:
def list_files(folder: Path, suffixes=(".csv", ".parquet", ".json")):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in suffixes])

def display_folder_files(folder: Path, title: str):
    files = list_files(folder)
    print("=" * 90)
    print(title)
    print("folder:", folder)
    print("num_files:", len(files))
    for p in files:
        print(" -", p.relative_to(folder))
    return files

def find_first_matching_file(folder: Path, name_candidates, prefer_parquet=False):
    files = list_files(folder)

    if not files:
        return None

    normalized_files = []
    for p in files:
        normalized_name = p.name.lower().replace("-", "_").replace(" ", "_")
        normalized_files.append((p, normalized_name))

    normalized_candidates = [
        c.lower().replace("-", "_").replace(" ", "_")
        for c in name_candidates
    ]

    # идём по кандидатам по порядку приоритета
    for cand in normalized_candidates:
        candidate_matches = [p for p, norm_name in normalized_files if cand in norm_name]

        if candidate_matches:
            candidate_matches = sorted(candidate_matches)

            if prefer_parquet:
                parquet_matches = [p for p in candidate_matches if p.suffix.lower() == ".parquet"]
                if parquet_matches:
                    return parquet_matches[0]

            return candidate_matches[0]

    return None

def read_table(path: Path):
    if path is None:
        return None
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    elif path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Неподдерживаемый формат файла: {path}")

## Инвентаризация артефактов после `02_data_preparation`
Сначала посмотрим, какие файлы лежат в `unified_corpus` и `cleaned`.

In [394]:
unified_files = display_folder_files(UNIFIED_CORPUS_DIR, "Файлы в unified_corpus")
cleaned_files = display_folder_files(CLEANED_DIR, "Файлы в cleaned")

Файлы в unified_corpus
folder: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/unified_corpus
num_files: 2
 - all_sources_standardized.csv
 - all_sources_standardized.parquet
Файлы в cleaned
folder: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned
num_files: 20
 - domain_aux_pool_prepared.csv
 - domain_aux_pool_prepared.parquet
 - domain_dialogues_aux_prepared.csv
 - domain_dialogues_aux_prepared.parquet
 - domain_dialogues_core_prepared.csv
 - domain_dialogues_core_prepared.parquet
 - domain_dialogues_prepared.csv
 - domain_dialogues_prepared.parquet
 - domain_text_pool_prepared.csv
 - domain_text_pool_prepared.parquet
 - education_feedback_document_level.csv
 - education_feedback_document_level.parquet
 - education_feedback_long.csv
 - education_feedback_long.parquet
 - emotion_aux_prepared.csv
 - emotion_aux_prepared.parquet
 - local_domain_template_prepared.csv
 - local_domain_template_prepared.parquet
 - sentiment_core_prepared.csv
 - sentime

## Поиск ключевых cleaned-артефактов
Нас сейчас интересуют прежде всего:
- общий unified слой;
- sentiment core;
- emotion auxiliary;
- education feedback;
- domain dialogues core;
- domain text pool;
- auxiliary domain pool;
- local domain template.

In [395]:
artifact_candidates = {
    "unified_all_sources": [
        "_all_sources_standardized",
        "all_sources_standardized",
        "all_sources",
        "unified_all_sources",
        "unified_corpus"
    ],
    "sentiment_core_prepared": [
        "sentiment_core_prepared",
        "sentiment_core"
    ],
    "emotion_aux_prepared": [
        "emotion_aux_prepared",
        "emotion_aux"
    ],
    "education_feedback_prepared": [
        "education_feedback_document_level",
        "education_feedback_prepared",
        "education_feedback"
    ],
    "education_feedback_long_prepared": [
        "education_feedback_long_prepared",
        "education_feedback_long"
    ],
    "domain_dialogues_core_prepared": [
        "domain_dialogues_core_prepared",
        "domain_dialogues_core"
    ],
    "domain_dialogues_aux_prepared": [
        "domain_dialogues_aux_prepared",
        "domain_dialogues_aux"
    ],
    "domain_dialogues_prepared": [
        "domain_dialogues_prepared",
        "domain_dialogues"
    ],
    "domain_text_pool": [
        "domain_text_pool_prepared",
        "domain_text_pool"
    ],
    "domain_aux_pool": [
        "domain_aux_pool_prepared",
        "domain_aux_pool"
    ],
    "local_domain_template_prepared": [
        "local_domain_template_prepared",
        "local_domain_template"
    ]
}

artifact_paths = {}

for artifact_name, candidates in artifact_candidates.items():
    if artifact_name == "unified_all_sources":
        artifact_paths[artifact_name] = find_first_matching_file(
            UNIFIED_CORPUS_DIR,
            candidates,
            prefer_parquet=True
        )
    else:
        artifact_paths[artifact_name] = find_first_matching_file(
            CLEANED_DIR,
            candidates,
            prefer_parquet=False
        )

artifact_paths_df = pd.DataFrame([
    {
        "artifact_name": k,
        "path": str(v) if v is not None else None,
        "exists": v is not None
    }
    for k, v in artifact_paths.items()
])

display(artifact_paths_df)

,artifact_name,path,exists
0,unified_all_sources,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/unified_corpus/all_sources_standardized.parquet,True
1,sentiment_core_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/sentiment_core_prepared.csv,True
2,emotion_aux_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/emotion_aux_prepared.csv,True
3,education_feedback_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/education_feedback_document_level.csv,True
4,education_feedback_long_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/education_feedback_long.csv,True
5,domain_dialogues_core_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_dialogues_core_prepared.csv,True
6,domain_dialogues_aux_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_dialogues_aux_prepared.csv,True
7,domain_dialogues_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_dialogues_prepared.csv,True
8,domain_text_pool,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_text_pool_prepared.csv,True
9,domain_aux_pool,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_aux_pool_prepared.csv,True


## Загрузка ключевых cleaned-артефактов
Если файл найден, загружаем его в DataFrame.

In [396]:
loaded_artifacts = {}

for artifact_name, path in artifact_paths.items():
    if path is not None:
        loaded_artifacts[artifact_name] = read_table(path)
    else:
        loaded_artifacts[artifact_name] = None

for artifact_name, df in loaded_artifacts.items():
    if df is None:
        print(f"{artifact_name}: NOT FOUND")
    else:
        print(f"{artifact_name}: shape={df.shape}")

unified_all_sources: shape=(121708, 14)
sentiment_core_prepared: shape=(34331, 14)
emotion_aux_prepared: shape=(9410, 14)
education_feedback_prepared: shape=(1274, 14)
education_feedback_long_prepared: shape=(24206, 14)
domain_dialogues_core_prepared: shape=(66045, 14)
domain_dialogues_aux_prepared: shape=(400, 14)
domain_dialogues_prepared: shape=(66445, 14)
domain_text_pool: shape=(67319, 14)
domain_aux_pool: shape=(400, 14)
local_domain_template_prepared: shape=(2, 14)


## Базовая проверка структуры загруженных артефактов
Смотрим:
- размеры таблиц;
- названия колонок;
- первые строки.

In [397]:
for artifact_name, df in loaded_artifacts.items():
    print("\n" + "=" * 100)
    print("ARTIFACT:", artifact_name)

    if df is None:
        print("Файл не найден")
        continue

    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head(3))


ARTIFACT: unified_all_sources
shape: (121708, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,<NA>,<NA>,<NA>,Прорвём информационную блокаду изнутри.,Прорвём информационную блокаду изнутри.,neutral,neutral,sentiment_3class_candidate,{}
1,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,<NA>,<NA>,<NA>,"Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.","Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.",negative,negative,sentiment_3class_candidate,{}
2,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,<NA>,<NA>,<NA>,"Кури-и тебя не укусит злая собака, потому что ты воняешь,кури-и тебя не ограбят,так как ты испугаешь грабителей отвратительным громким кашлем, кури - и ты не умрешь от старости, просто не доживешь...","Кури-и тебя не укусит злая собака, потому что ты воняешь,кури-и тебя не ограбят,так как ты испугаешь грабителей отвратительным громким кашлем, кури - и ты не умрешь от старости, просто не доживешь...",skip,<NA>,sentiment_3class_candidate,{}



ARTIFACT: sentiment_core_prepared
shape: (34331, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,NaN,NaN,NaN,Прорвём информационную блокаду изнутри.,Прорвём информационную блокаду изнутри.,neutral,neutral,sentiment_3class_candidate,{}
1,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,NaN,NaN,NaN,"Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.","Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.",negative,negative,sentiment_3class_candidate,{}
2,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,NaN,NaN,NaN,"Есть 3 типа людей: Умные, которые делают все сразу. Умные, которые откладывают на потом и доделывают. И Я...сначала впадлу потом поздно","Есть 3 типа людей: Умные, которые делают все сразу. Умные, которые откладывают на потом и доделывают. И Я...сначала впадлу потом поздно",neutral,neutral,sentiment_3class_candidate,{}



ARTIFACT: emotion_aux_prepared
shape: (9410, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


...

,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,student_feedback_ru,domain_education,domain,test.csv,test,NaN,NaN,student,"Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ...","Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т..."
1,student_feedback_ru,domain_education,domain,test.csv,test,NaN,NaN,student,"Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем.","Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем.",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т..."
2,student_feedback_ru,domain_education,domain,test.csv,test,NaN,NaN,student,"Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в...","Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 1, ""видео-уроки"": 1, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 1, ""практики__семинары"": 0, ""т..."



ARTIFACT: domain_aux_pool
shape: (400, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,instructional_dialogues_ru,domain_dialogue_aux,aux_domain,instructional_dialogues_ru.csv,unknown,1,0,student,Как мне topic 1?,Как мне topic 1?,NaN,NaN,unlabeled_dialogue,"{""topic"": ""Topic 1"", ""goal"": ""Пользователь узнаёт, как topic 1.""}"
1,instructional_dialogues_ru,domain_dialogue_aux,aux_domain,instructional_dialogues_ru.csv,unknown,1,1,tutor_analog,"Вот шаги, как topic 1.","Вот шаги, как topic 1.",NaN,NaN,unlabeled_dialogue,"{""topic"": ""Topic 1"", ""goal"": ""Пользователь узнаёт, как topic 1.""}"
2,instructional_dialogues_ru,domain_dialogue_aux,aux_domain,instructional_dialogues_ru.csv,unknown,1,2,student,Спасибо!,Спасибо!,NaN,NaN,unlabeled_dialogue,"{""topic"": ""Topic 1"", ""goal"": ""Пользователь узнаёт, как topic 1.""}"



ARTIFACT: local_domain_template_prepared
shape: (2, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,local_domain_ru,local_domain,future_core,local_domain_ru_template.csv,template,conv_0001,0,student,"Мне сложно понять, как выбрать следующую дисциплину.","Мне сложно понять, как выбрать следующую дисциплину.",NaN,NaN,unlabeled_dialogue,"{""meta"": NaN}"
1,local_domain_ru,local_domain,future_core,local_domain_ru_template.csv,template,conv_0001,1,tutor,Давай посмотрим на твою цель и текущую траекторию.,Давай посмотрим на твою цель и текущую траекторию.,NaN,NaN,unlabeled_dialogue,"{""meta"": NaN}"


## Контроль ключевых полей
Проверяем, что в основных таблицах есть хотя бы базовые поля:
- `dataset_name`
- `text_clean`
- `target_label` (где должна быть supervised-разметка)
- `speaker_role`
- `conversation_id`
- `turn_id`

In [398]:
key_columns = ["dataset_name", "text_clean", "target_label", "speaker_role", "conversation_id", "turn_id"]

check_rows = []

for artifact_name, df in loaded_artifacts.items():
    if df is None:
        continue

    row = {"artifact_name": artifact_name, "rows": len(df)}
    for col in key_columns:
        row[col] = col in df.columns
    check_rows.append(row)

structure_check_df = pd.DataFrame(check_rows)
display(structure_check_df)

,artifact_name,rows,dataset_name,text_clean,target_label,speaker_role,conversation_id,turn_id
0,unified_all_sources,121708,True,True,True,True,True,True
1,sentiment_core_prepared,34331,True,True,True,True,True,True
2,emotion_aux_prepared,9410,True,True,True,True,True,True
3,education_feedback_prepared,1274,True,True,True,True,True,True
4,education_feedback_long_prepared,24206,True,True,True,True,True,True
5,domain_dialogues_core_prepared,66045,True,True,True,True,True,True
6,domain_dialogues_aux_prepared,400,True,True,True,True,True,True
7,domain_dialogues_prepared,66445,True,True,True,True,True,True
8,domain_text_pool,67319,True,True,True,True,True,True
9,domain_aux_pool,400,True,True,True,True,True,True


## Первая содержательная сводка
Сейчас нам важно увидеть:
- какие артефакты уже содержат `target_label`;
- какие артефакты являются доменно близкими, но пока неразмеченными;
- сколько в них строк.

In [399]:
summary_rows = []

for artifact_name, df in loaded_artifacts.items():
    if df is None:
        continue

    row = {
        "artifact_name": artifact_name,
        "rows": len(df),
        "has_target_label_column": "target_label" in df.columns,
        "non_null_target_label": int(df["target_label"].notna().sum()) if "target_label" in df.columns else 0,
        "has_speaker_role": "speaker_role" in df.columns,
        "has_conversation_id": "conversation_id" in df.columns,
        "has_turn_id": "turn_id" in df.columns,
        "num_datasets_inside": int(df["dataset_name"].nunique()) if "dataset_name" in df.columns else None
    }
    summary_rows.append(row)

artifact_summary_df = pd.DataFrame(summary_rows).sort_values(by="rows", ascending=False)
display(artifact_summary_df)

,artifact_name,rows,has_target_label_column,non_null_target_label,has_speaker_role,has_conversation_id,has_turn_id,num_datasets_inside
0,unified_all_sources,121708,True,34331,True,True,True,7
8,domain_text_pool,67319,True,0,True,True,True,2
7,domain_dialogues_prepared,66445,True,0,True,True,True,2
5,domain_dialogues_core_prepared,66045,True,0,True,True,True,1
1,sentiment_core_prepared,34331,True,34331,True,True,True,2
4,education_feedback_long_prepared,24206,True,0,True,True,True,1
2,emotion_aux_prepared,9410,True,0,True,True,True,1
3,education_feedback_prepared,1274,True,0,True,True,True,1
6,domain_dialogues_aux_prepared,400,True,0,True,True,True,1
9,domain_aux_pool,400,True,0,True,True,True,1


## Контроль по классам в sentiment-ядре
Эта ячейка нужна, чтобы понять, действительно ли `sentiment_core_prepared` уже готов как основа для 3-классовой задачи.

In [400]:
sentiment_core_df = loaded_artifacts.get("sentiment_core_prepared")

if sentiment_core_df is None:
    print("sentiment_core_prepared не найден")
else:
    print("shape:", sentiment_core_df.shape)
    if "target_label" in sentiment_core_df.columns:
        display(
            sentiment_core_df["target_label"]
            .fillna("<NA>")
            .value_counts(dropna=False)
            .rename_axis("target_label")
            .reset_index(name="count")
        )

    if "dataset_name" in sentiment_core_df.columns:
        display(
            sentiment_core_df.groupby(["dataset_name", "target_label"])
            .size()
            .reset_index(name="count")
            .sort_values(["dataset_name", "count"], ascending=[True, False])
        )

shape: (34331, 14)


,target_label,count
0,neutral,18061
1,positive,9060
2,negative,7210


,dataset_name,target_label,count
1,rusentiment,neutral,12720
2,rusentiment,positive,6646
0,rusentiment,negative,3912
4,rusentitweet,neutral,5341
3,rusentitweet,negative,3298
5,rusentitweet,positive,2414


## Контроль доменного пула
Посмотрим, из чего состоит `domain_text_pool`.

In [401]:
domain_text_pool_df = loaded_artifacts.get("domain_text_pool")

if domain_text_pool_df is None:
    print("domain_text_pool не найден")
else:
    print("shape:", domain_text_pool_df.shape)

    if "dataset_name" in domain_text_pool_df.columns:
        display(
            domain_text_pool_df["dataset_name"]
            .value_counts(dropna=False)
            .rename_axis("dataset_name")
            .reset_index(name="count")
        )

    preview_cols = [c for c in ["dataset_name", "speaker_role", "text_clean", "target_label"] if c in domain_text_pool_df.columns]
    display(domain_text_pool_df[preview_cols].head(10))

shape: (67319, 14)


,dataset_name,count
0,teacher_student_dialogues_ru,66045
1,student_feedback_ru,1274


,dataset_name,speaker_role,text_clean,target_label
0,student_feedback_ru,student,"Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ...",NaN
1,student_feedback_ru,student,"Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем.",NaN
2,student_feedback_ru,student,"Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в...",NaN
3,student_feedback_ru,student,"Отличный электив! Преподаватель - умная, начитанная, любит работать со студентами, любит свою работу в целом. Курс позволяет изучить лексику по теме психология и логопедия, ну и соответственно узн...",NaN
4,student_feedback_ru,student,"преподша- Марандина Елена Леонидовна. Электив проходил в соцгуме. Электив нормальный, смотрите презентации, записываете лекции и иногда делаете мини-проверочные. По ощущениям-как подготовка к ОГЭ ...",NaN
5,student_feedback_ru,student,"Вам это не надо, не берите этот электив",NaN
6,student_feedback_ru,student,"Здесь одни семинары, без высшего участия тут пара не пройдёт, но самое главное - интересно. Педагог максимально пытается заявлечь, и у неё это получается. Она максимально показывает свое мастерств...",NaN
7,student_feedback_ru,student,"Хорошая возможность узнать о странах побольше! Очень доброжелательный преподаватель, критика только для помощи! На элективе есть разные варианты набора баллов, что очень удобно.",NaN
8,student_feedback_ru,student,"Очень рада, что выбрала именно этот электив. Прекраснейшая преподавательница, даёт реально много полезной информации, вдохновляет. На парах было много практики говорения, именно Speaking CAE. Со в...",NaN
9,student_feedback_ru,student,"Многим курс понравился, в восторге от него. Но лично для меня он не совсем оправдал ожидания. Кто любит поговорить на темы о психологии, вам туда и получите максимальные баллы. Кто не особо открыт...",NaN


## Промежуточное решение по контуру данных

На текущем шаге фиксируем:

- `sentiment_core_prepared` — основное supervised sentiment-ядро;
- `domain_text_pool` — основной доменно близкий пул текстов;
- `domain_aux_pool` — вспомогательный доменный пул;
- `emotion_aux_prepared` — отдельный эмоциональный auxiliary-источник, не смешиваем автоматически с 3-классовой задачей;
- `local_domain_template_prepared` — шаблон будущего собственного корпуса, в основной эксперимент пока не включается.


In [402]:
sentiment_core_df = loaded_artifacts["sentiment_core_prepared"].copy()
domain_text_pool_df = loaded_artifacts["domain_text_pool"].copy()
domain_aux_pool_df = loaded_artifacts["domain_aux_pool"].copy()
emotion_aux_df = loaded_artifacts["emotion_aux_prepared"].copy()
local_domain_template_df = loaded_artifacts["local_domain_template_prepared"].copy()

print("sentiment_core_df       :", sentiment_core_df.shape)
print("domain_text_pool_df     :", domain_text_pool_df.shape)
print("domain_aux_pool_df      :", domain_aux_pool_df.shape)
print("emotion_aux_df          :", emotion_aux_df.shape)
print("local_domain_template_df:", local_domain_template_df.shape)

sentiment_core_df       : (34331, 14)
domain_text_pool_df     : (67319, 14)
domain_aux_pool_df      : (400, 14)
emotion_aux_df          : (9410, 14)
local_domain_template_df: (2, 14)


## Первичный аудит доменного пула

Сейчас интересует:
- распределение по источникам;
- распределение по ролям;
- длина текстов;
- доля пустых и очень коротких строк;
- примерный объём дублей.

Это нужно, чтобы понять, какой поднабор можно отдавать на ручную разметку.

In [403]:
import re

def normalize_spaces(text):
    if pd.isna(text):
        return None
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None

def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

domain_text_pool_df["text_clean"] = domain_text_pool_df["text_clean"].map(normalize_spaces)
domain_text_pool_df["char_len"] = domain_text_pool_df["text_clean"].fillna("").str.len()
domain_text_pool_df["word_len"] = domain_text_pool_df["text_clean"].map(count_words)

domain_text_pool_df["text_norm_for_dup"] = (
    domain_text_pool_df["text_clean"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

domain_text_pool_df["is_duplicate_norm"] = domain_text_pool_df.duplicated("text_norm_for_dup", keep=False)
domain_text_pool_df["is_empty_text"] = domain_text_pool_df["text_clean"].isna() | (domain_text_pool_df["text_clean"].str.strip() == "")
domain_text_pool_df["is_too_short_words"] = domain_text_pool_df["word_len"] <= 2
domain_text_pool_df["is_too_short_chars"] = domain_text_pool_df["char_len"] < 15

In [404]:
domain_qc_summary = pd.DataFrame([
    {
        "rows_total": len(domain_text_pool_df),
        "empty_text_rows": int(domain_text_pool_df["is_empty_text"].sum()),
        "too_short_words_rows": int(domain_text_pool_df["is_too_short_words"].sum()),
        "too_short_chars_rows": int(domain_text_pool_df["is_too_short_chars"].sum()),
        "duplicate_norm_rows": int(domain_text_pool_df["is_duplicate_norm"].sum()),
        "mean_char_len": float(domain_text_pool_df["char_len"].mean()),
        "median_char_len": float(domain_text_pool_df["char_len"].median()),
        "mean_word_len": float(domain_text_pool_df["word_len"].mean()),
        "median_word_len": float(domain_text_pool_df["word_len"].median()),
    }
])

display(domain_qc_summary)

,rows_total,empty_text_rows,too_short_words_rows,too_short_chars_rows,duplicate_norm_rows,mean_char_len,median_char_len,mean_word_len,median_word_len
0,67319,0,11,7,76,332.237214,245.0,46.812163,36.0


In [405]:
print("Распределение по источникам:")
display(
    domain_text_pool_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

print("Распределение по ролям:")
display(
    domain_text_pool_df["speaker_role"]
    .fillna("<NA>")
    .value_counts(dropna=False)
    .rename_axis("speaker_role")
    .reset_index(name="count")
)

print("Распределение по источникам и ролям:")
display(
    domain_text_pool_df
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "count"], ascending=[True, False])
)

Распределение по источникам:


,dataset_name,count
0,teacher_student_dialogues_ru,66045
1,student_feedback_ru,1274


Распределение по ролям:


,speaker_role,count
0,student,34828
1,teacher_analog,32491


Распределение по источникам и ролям:


,dataset_name,speaker_role,count
0,student_feedback_ru,student,1274
1,teacher_student_dialogues_ru,student,33554
2,teacher_student_dialogues_ru,teacher_analog,32491


In [406]:
print("Подозрительные строки по длине:")
display(
    domain_text_pool_df.loc[
        domain_text_pool_df["is_too_short_words"] | domain_text_pool_df["is_too_short_chars"],
        ["dataset_name", "speaker_role", "char_len", "word_len", "text_clean"]
    ].head(30)
)

print("Примеры дубликатов:")
display(
    domain_text_pool_df.loc[
        domain_text_pool_df["is_duplicate_norm"],
        ["dataset_name", "speaker_role", "char_len", "word_len", "text_clean"]
    ].head(30)
)

Подозрительные строки по длине:


,dataset_name,speaker_role,char_len,word_len,text_clean
107,student_feedback_ru,student,14,2,духота ебанная
176,student_feedback_ru,student,13,2,препод хуесос
255,student_feedback_ru,student,11,2,Кайфы ловлю
294,student_feedback_ru,student,5,1,Круто
446,student_feedback_ru,student,12,2,круто всё!!!
819,student_feedback_ru,student,19,2,прекрасный электив!
862,student_feedback_ru,student,10,1,прекрасно!
889,student_feedback_ru,student,16,2,Классные преподы
1165,student_feedback_ru,student,16,2,Легко закрыться!
1171,student_feedback_ru,student,8,2,НЕ ФИГНЯ


Примеры дубликатов:


,dataset_name,speaker_role,char_len,word_len,text_clean
601,student_feedback_ru,student,285,40,"Это тот электив, где преподователь доступно преподносит теоретический материал, и наглядно показывает применение в практике. Будет полезно, даже если вы не изучали сетевые технологии раньше, матер..."
818,student_feedback_ru,student,285,40,"Это тот электив, где преподователь доступно преподносит теоретический материал, и наглядно показывает применение в практике. Будет полезно, даже если вы не изучали сетевые технологии раньше, матер..."
1127,student_feedback_ru,student,315,45,"Попала на электив случайно, подойдет только тем, кто увлекается историей, сбором архивных данных. Без базовых знаний и понимания истории здесь делать нечего. Каждый семинар проходил по типу: ответ..."
1260,student_feedback_ru,student,315,45,"Попала на электив случайно, подойдет только тем, кто увлекается историей, сбором архивных данных. Без базовых знаний и понимания истории здесь делать нечего. Каждый семинар проходил по типу: ответ..."
25788,teacher_student_dialogues_ru,student,129,19,"Я слышал, что психоделическая терапия может лечить депрессию, но это звучит как фантастика. Какие научные гипотезы за этим стоят?"
30609,teacher_student_dialogues_ru,student,129,19,"Я слышал, что психоделическая терапия может лечить депрессию, но это звучит как фантастика. Какие научные гипотезы за этим стоят?"
37101,teacher_student_dialogues_ru,student,136,19,"Интересно, как семиотика помогает понять, как язык передает научные знания? И почему русский и английский справляются с этим по-разному?"
38179,teacher_student_dialogues_ru,student,66,10,"Понятно, а метафоры? Как они отличаются в описании ядерной угрозы?"
38433,teacher_student_dialogues_ru,student,137,22,"Интересно, как семиотика помогает понять, как мы передаем научные знания через язык? И есть ли разница между русским и английским в этом?"
38435,teacher_student_dialogues_ru,student,79,13,А можно пример из биологии? Как различия в языках влияют на понимание эволюции?


In [407]:
print("Примеры из teacher_student_dialogues_ru:")
display(
    domain_text_pool_df.loc[
        domain_text_pool_df["dataset_name"] == "teacher_student_dialogues_ru",
        ["speaker_role", "char_len", "word_len", "text_clean"]
    ].head(15)
)

print("Примеры из student_feedback_ru:")
display(
    domain_text_pool_df.loc[
        domain_text_pool_df["dataset_name"] == "student_feedback_ru",
        ["speaker_role", "char_len", "word_len", "text_clean"]
    ].head(15)
)

Примеры из teacher_student_dialogues_ru:


,speaker_role,char_len,word_len,text_clean
1274,student,156,23,"Привет! Я всегда интересовался сленгом в разных языках. Как ты думаешь, в чем основные различия между русским и английским сленгом? Можешь привести примеры?"
1275,teacher_analog,474,68,"Конечно! Сленг в обоих языках эволюционирует быстро, но английский сленг часто заимствует слова из других культур из-за глобализации, в то время как русский сленг больше опирается на внутренние шу..."
1276,student,110,18,"Интересно! А что насчет молодежного сленга? В английском я слышал 'lit' или 'sus', как это сравнить с русским?"
1277,teacher_analog,596,89,"Да, молодежный сленг — это отдельная тема. 'Lit' в английском значит 'круто' или 'весело', от 'light it up', подразумевая что-то зажигательное. В русском аналог — 'зажечь' или 'огонь', что тоже св..."
1278,student,202,34,"Привет! Я изучаю русский язык и заметил, что частицы вроде 'не' или 'ли' сильно влияют на смысл. Можно ли разобрать это на примерах из русских праздников и традиций? И как это отличается от англий..."
1279,teacher_analog,488,66,"Конечно, давай разберёмся! Возьмём частицу 'ли', которая используется для образования вопросов в русском, в отличие от английского, где вопросы часто строятся с помощью вспомогательных глаголов. Н..."
1280,student,136,21,"Интересно! А что насчёт отрицательной частицы 'не'? Как она работает в предложениях о традициях, и чем отличается от 'not' в английском?"
1281,teacher_analog,547,79,"Отличный вопрос! Частица 'не' в русском может быть как префиксом, меняющим значение глагола, так и отдельным словом для отрицания. Например, в традиции Нового года: 'Не загадывай желания под бой к..."
1282,student,141,23,"Привет! Я заметил, что в магазинах люди часто используют сленг, особенно молодежь. Как это работает на русском и английском, и в чем разница?"
1283,teacher_analog,455,71,"Отличный вопрос! В русском сленге в магазине, например, вместо 'купить' могут сказать 'забрать' или 'приобрести по акции' как 'срубить по дешевке'. На английском это 'score a deal' или 'grab it on..."


Примеры из student_feedback_ru:


,speaker_role,char_len,word_len,text_clean
0,student,1044,176,"Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ..."
1,student,142,19,"Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем."
2,student,611,98,"Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в..."
3,student,738,99,"Отличный электив! Преподаватель - умная, начитанная, любит работать со студентами, любит свою работу в целом. Курс позволяет изучить лексику по теме психология и логопедия, ну и соответственно узн..."
4,student,614,85,"преподша- Марандина Елена Леонидовна. Электив проходил в соцгуме. Электив нормальный, смотрите презентации, записываете лекции и иногда делаете мини-проверочные. По ощущениям-как подготовка к ОГЭ ..."
5,student,39,8,"Вам это не надо, не берите этот электив"
6,student,359,50,"Здесь одни семинары, без высшего участия тут пара не пройдёт, но самое главное - интересно. Педагог максимально пытается заявлечь, и у неё это получается. Она максимально показывает свое мастерств..."
7,student,177,23,"Хорошая возможность узнать о странах побольше! Очень доброжелательный преподаватель, критика только для помощи! На элективе есть разные варианты набора баллов, что очень удобно."
8,student,355,49,"Очень рада, что выбрала именно этот электив. Прекраснейшая преподавательница, даёт реально много полезной информации, вдохновляет. На парах было много практики говорения, именно Speaking CAE. Со в..."
9,student,221,36,"Многим курс понравился, в восторге от него. Но лично для меня он не совсем оправдал ожидания. Кто любит поговорить на темы о психологии, вам туда и получите максимальные баллы. Кто не особо открыт..."


## Формируем кандидатный доменный пул для ручной разметки

На предыдущем шаге убедились, что `domain_text_pool` в целом качественный, но перед ручной разметкой всё равно нужно убрать:
- пустые строки;
- слишком короткие строки;
- нормализованные дубли.

После этого получим `filtered_domain_pool_df` — кандидатный пул для дальнейшей квотированной выборки.

In [408]:
filtered_domain_pool_df = domain_text_pool_df.loc[
    (~domain_text_pool_df["is_empty_text"]) &
    (~domain_text_pool_df["is_too_short_words"]) &
    (~domain_text_pool_df["is_too_short_chars"]) &
    (~domain_text_pool_df["is_duplicate_norm"])
].copy()

print("domain_text_pool_df      :", domain_text_pool_df.shape)
print("filtered_domain_pool_df  :", filtered_domain_pool_df.shape)
print("removed_rows             :", len(domain_text_pool_df) - len(filtered_domain_pool_df))

domain_text_pool_df      : (67319, 21)
filtered_domain_pool_df  : (67232, 21)
removed_rows             : 87


In [409]:
filtering_report_df = pd.DataFrame([
    {
        "rows_before": len(domain_text_pool_df),
        "rows_after": len(filtered_domain_pool_df),
        "removed_total": len(domain_text_pool_df) - len(filtered_domain_pool_df),
        "removed_share": (len(domain_text_pool_df) - len(filtered_domain_pool_df)) / len(domain_text_pool_df),
        "empty_text_rows": int(domain_text_pool_df["is_empty_text"].sum()),
        "too_short_words_rows": int(domain_text_pool_df["is_too_short_words"].sum()),
        "too_short_chars_rows": int(domain_text_pool_df["is_too_short_chars"].sum()),
        "duplicate_norm_rows": int(domain_text_pool_df["is_duplicate_norm"].sum()),
    }
])

display(filtering_report_df)

,rows_before,rows_after,removed_total,removed_share,empty_text_rows,too_short_words_rows,too_short_chars_rows,duplicate_norm_rows
0,67319,67232,87,0.001292,0,11,7,76


## Проверяем распределение после фильтрации

Важно убедиться, что после удаления коротких строк и дублей:
- не исчез какой-то источник;
- не нарушился ролевой баланс;
- `student_feedback_ru` по-прежнему остаётся в рабочем контуре.

In [410]:
print("Распределение по источникам после фильтрации:")
display(
    filtered_domain_pool_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

print("Распределение по ролям после фильтрации:")
display(
    filtered_domain_pool_df["speaker_role"]
    .fillna("<NA>")
    .value_counts(dropna=False)
    .rename_axis("speaker_role")
    .reset_index(name="count")
)

print("Распределение по источникам и ролям после фильтрации:")
display(
    filtered_domain_pool_df
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "count"], ascending=[True, False])
)

Распределение по источникам после фильтрации:


,dataset_name,count
0,teacher_student_dialogues_ru,65973
1,student_feedback_ru,1259


Распределение по ролям после фильтрации:


,speaker_role,count
0,student,34741
1,teacher_analog,32491


Распределение по источникам и ролям после фильтрации:


,dataset_name,speaker_role,count
0,student_feedback_ru,student,1259
1,teacher_student_dialogues_ru,student,33482
2,teacher_student_dialogues_ru,teacher_analog,32491


In [411]:
print("Примеры после фильтрации: teacher_student_dialogues_ru")
display(
    filtered_domain_pool_df.loc[
        filtered_domain_pool_df["dataset_name"] == "teacher_student_dialogues_ru",
        ["speaker_role", "char_len", "word_len", "text_clean"]
    ].head(10)
)

print("Примеры после фильтрации: student_feedback_ru")
display(
    filtered_domain_pool_df.loc[
        filtered_domain_pool_df["dataset_name"] == "student_feedback_ru",
        ["speaker_role", "char_len", "word_len", "text_clean"]
    ].head(10)
)

Примеры после фильтрации: teacher_student_dialogues_ru


,speaker_role,char_len,word_len,text_clean
1274,student,156,23,"Привет! Я всегда интересовался сленгом в разных языках. Как ты думаешь, в чем основные различия между русским и английским сленгом? Можешь привести примеры?"
1275,teacher_analog,474,68,"Конечно! Сленг в обоих языках эволюционирует быстро, но английский сленг часто заимствует слова из других культур из-за глобализации, в то время как русский сленг больше опирается на внутренние шу..."
1276,student,110,18,"Интересно! А что насчет молодежного сленга? В английском я слышал 'lit' или 'sus', как это сравнить с русским?"
1277,teacher_analog,596,89,"Да, молодежный сленг — это отдельная тема. 'Lit' в английском значит 'круто' или 'весело', от 'light it up', подразумевая что-то зажигательное. В русском аналог — 'зажечь' или 'огонь', что тоже св..."
1278,student,202,34,"Привет! Я изучаю русский язык и заметил, что частицы вроде 'не' или 'ли' сильно влияют на смысл. Можно ли разобрать это на примерах из русских праздников и традиций? И как это отличается от англий..."
1279,teacher_analog,488,66,"Конечно, давай разберёмся! Возьмём частицу 'ли', которая используется для образования вопросов в русском, в отличие от английского, где вопросы часто строятся с помощью вспомогательных глаголов. Н..."
1280,student,136,21,"Интересно! А что насчёт отрицательной частицы 'не'? Как она работает в предложениях о традициях, и чем отличается от 'not' в английском?"
1281,teacher_analog,547,79,"Отличный вопрос! Частица 'не' в русском может быть как префиксом, меняющим значение глагола, так и отдельным словом для отрицания. Например, в традиции Нового года: 'Не загадывай желания под бой к..."
1282,student,141,23,"Привет! Я заметил, что в магазинах люди часто используют сленг, особенно молодежь. Как это работает на русском и английском, и в чем разница?"
1283,teacher_analog,455,71,"Отличный вопрос! В русском сленге в магазине, например, вместо 'купить' могут сказать 'забрать' или 'приобрести по акции' как 'срубить по дешевке'. На английском это 'score a deal' или 'grab it on..."


Примеры после фильтрации: student_feedback_ru


,speaker_role,char_len,word_len,text_clean
0,student,1044,176,"Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ..."
1,student,142,19,"Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем."
2,student,611,98,"Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в..."
3,student,738,99,"Отличный электив! Преподаватель - умная, начитанная, любит работать со студентами, любит свою работу в целом. Курс позволяет изучить лексику по теме психология и логопедия, ну и соответственно узн..."
4,student,614,85,"преподша- Марандина Елена Леонидовна. Электив проходил в соцгуме. Электив нормальный, смотрите презентации, записываете лекции и иногда делаете мини-проверочные. По ощущениям-как подготовка к ОГЭ ..."
5,student,39,8,"Вам это не надо, не берите этот электив"
6,student,359,50,"Здесь одни семинары, без высшего участия тут пара не пройдёт, но самое главное - интересно. Педагог максимально пытается заявлечь, и у неё это получается. Она максимально показывает свое мастерств..."
7,student,177,23,"Хорошая возможность узнать о странах побольше! Очень доброжелательный преподаватель, критика только для помощи! На элективе есть разные варианты набора баллов, что очень удобно."
8,student,355,49,"Очень рада, что выбрала именно этот электив. Прекраснейшая преподавательница, даёт реально много полезной информации, вдохновляет. На парах было много практики говорения, именно Speaking CAE. Со в..."
9,student,221,36,"Многим курс понравился, в восторге от него. Но лично для меня он не совсем оправдал ожидания. Кто любит поговорить на темы о психологии, вам туда и получите максимальные баллы. Кто не особо открыт..."


## Подготовка к квотированной выборке

Поскольку `teacher_student_dialogues_ru` сильно больше, чем `student_feedback_ru`,
случайная выборка без квот приведёт к почти полному исчезновению второго источника.

Поэтому дальше будем смотреть не просто общий объём, а доступный объём по группам:
- `teacher_student_dialogues_ru / student`
- `teacher_student_dialogues_ru / teacher_analog`
- `student_feedback_ru / student`

In [412]:
available_groups_df = (
    filtered_domain_pool_df
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="available_rows")
    .sort_values(["dataset_name", "available_rows"], ascending=[True, False])
)

display(available_groups_df)

,dataset_name,speaker_role,available_rows
0,student_feedback_ru,student,1259
1,teacher_student_dialogues_ru,student,33482
2,teacher_student_dialogues_ru,teacher_analog,32491


In [413]:
available_groups_df["share_in_filtered_pool"] = (
    available_groups_df["available_rows"] / available_groups_df["available_rows"].sum()
)

display(available_groups_df)

,dataset_name,speaker_role,available_rows,share_in_filtered_pool
0,student_feedback_ru,student,1259,0.018726
1,teacher_student_dialogues_ru,student,33482,0.498007
2,teacher_student_dialogues_ru,teacher_analog,32491,0.483267


## Черновые сценарии размера батча на ручную разметку

Пока не фиксируем окончательный размер, а просто смотрим,
как могли бы выглядеть квоты для нескольких разумных вариантов:
- 600
- 900
- 1200
- 1500


In [414]:
batch_sizes = [600, 900, 1200, 1500]

quota_scenarios = []

for batch_size in batch_sizes:
    for _, row in available_groups_df.iterrows():
        quota_scenarios.append({
            "batch_size": batch_size,
            "dataset_name": row["dataset_name"],
            "speaker_role": row["speaker_role"],
            "available_rows": int(row["available_rows"]),
            "share_in_filtered_pool": float(row["share_in_filtered_pool"]),
            "proportional_quota_raw": row["available_rows"] / available_groups_df["available_rows"].sum() * batch_size,
        })

quota_scenarios_df = pd.DataFrame(quota_scenarios)
quota_scenarios_df["proportional_quota"] = quota_scenarios_df["proportional_quota_raw"].round().astype(int)

display(
    quota_scenarios_df[
        ["batch_size", "dataset_name", "speaker_role", "available_rows", "share_in_filtered_pool", "proportional_quota"]
    ]
)

,batch_size,dataset_name,speaker_role,available_rows,share_in_filtered_pool,proportional_quota
0,600,student_feedback_ru,student,1259,0.018726,11
1,600,teacher_student_dialogues_ru,student,33482,0.498007,299
2,600,teacher_student_dialogues_ru,teacher_analog,32491,0.483267,290
3,900,student_feedback_ru,student,1259,0.018726,17
4,900,teacher_student_dialogues_ru,student,33482,0.498007,448
5,900,teacher_student_dialogues_ru,teacher_analog,32491,0.483267,435
6,1200,student_feedback_ru,student,1259,0.018726,22
7,1200,teacher_student_dialogues_ru,student,33482,0.498007,598
8,1200,teacher_student_dialogues_ru,teacher_analog,32491,0.483267,580
9,1500,student_feedback_ru,student,1259,0.018726,28


## Переходим к ручным квотам

Пропорциональные квоты нам не подходят, потому что они почти исключают `student_feedback_ru`.
Поэтому фиксируем ручной сценарий батча на разметку.

Выбранный сценарий:
- `teacher_student_dialogues_ru / student` - 300
- `teacher_student_dialogues_ru / teacher_analog` - 300
- `student_feedback_ru / student` - 300

Итого:
- 900 строк на ручную разметку.

In [415]:
RANDOM_STATE = 42
BATCH_NAME = "labeling_batch_v2_domain_ru"

quota_plan = pd.DataFrame([
    {
        "dataset_name": "teacher_student_dialogues_ru",
        "speaker_role": "student",
        "target_quota": 300
    },
    {
        "dataset_name": "teacher_student_dialogues_ru",
        "speaker_role": "teacher_analog",
        "target_quota": 300
    },
    {
        "dataset_name": "student_feedback_ru",
        "speaker_role": "student",
        "target_quota": 300
    },
])

display(quota_plan)

,dataset_name,speaker_role,target_quota
0,teacher_student_dialogues_ru,student,300
1,teacher_student_dialogues_ru,teacher_analog,300
2,student_feedback_ru,student,300


## Подготавливаем пул по квотам

Перед выборкой нужно проверить:
- хватает ли строк в каждой группе;
- как будем контролировать разнообразие.

Для `teacher_student_dialogues_ru` желательно не брать слишком много строк из одной и той же беседы, поэтому введём ограничение:
- не более одной строки на `conversation_id` внутри каждой ролевой группы.

Для `student_feedback_ru` такого ограничения не требуется,
потому что это не диалоговый корпус.

In [416]:
group_availability_check_df = (
    filtered_domain_pool_df
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="available_rows")
)

quota_check_df = quota_plan.merge(
    group_availability_check_df,
    on=["dataset_name", "speaker_role"],
    how="left"
)

quota_check_df["enough_rows"] = quota_check_df["available_rows"] >= quota_check_df["target_quota"]

display(quota_check_df)

,dataset_name,speaker_role,target_quota,available_rows,enough_rows
0,teacher_student_dialogues_ru,student,300,33482,True
1,teacher_student_dialogues_ru,teacher_analog,300,32491,True
2,student_feedback_ru,student,300,1259,True


## Ограничение по разнообразию диалогов

Для `teacher_student_dialogues_ru` берём не более одной строки на `conversation_id`
внутри каждой ролевой группы.

Это нужно, чтобы батч не оказался переполнен репликами из одних и тех же разговоров.

In [417]:
dialogue_pool_df = filtered_domain_pool_df.loc[
    filtered_domain_pool_df["dataset_name"] == "teacher_student_dialogues_ru"
].copy()

feedback_pool_df = filtered_domain_pool_df.loc[
    filtered_domain_pool_df["dataset_name"] == "student_feedback_ru"
].copy()

print("dialogue_pool_df:", dialogue_pool_df.shape)
print("feedback_pool_df:", feedback_pool_df.shape)

dialogue_unique_conv_df = (
    dialogue_pool_df
    .drop_duplicates(subset=["dataset_name", "speaker_role", "conversation_id"])
    .copy()
)

print("dialogue_unique_conv_df:", dialogue_unique_conv_df.shape)

dialogue_pool_df: (65973, 21)
feedback_pool_df: (1259, 21)
dialogue_unique_conv_df: (19999, 21)


In [418]:
print("После ограничения one-row-per-conversation:")
display(
    dialogue_unique_conv_df
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="available_rows_after_conv_limit")
    .sort_values(["dataset_name", "available_rows_after_conv_limit"], ascending=[True, False])
)

После ограничения one-row-per-conversation:


,dataset_name,speaker_role,available_rows_after_conv_limit
1,teacher_student_dialogues_ru,teacher_analog,10000
0,teacher_student_dialogues_ru,student,9999


## Квотированная выборка

Теперь собираем батч:
- для `teacher_student_dialogues_ru` — из `dialogue_unique_conv_df`;
- для `student_feedback_ru` — из `feedback_pool_df`.

Используем фиксированный `random_state`, чтобы выборка была воспроизводимой.

In [419]:
sampled_parts = []

for _, quota_row in quota_plan.iterrows():
    dataset_name = quota_row["dataset_name"]
    speaker_role = quota_row["speaker_role"]
    target_quota = int(quota_row["target_quota"])

    if dataset_name == "teacher_student_dialogues_ru":
        source_df = dialogue_unique_conv_df
    elif dataset_name == "student_feedback_ru":
        source_df = feedback_pool_df
    else:
        raise ValueError(f"Неожиданный dataset_name: {dataset_name}")

    candidate_df = source_df.loc[
        (source_df["dataset_name"] == dataset_name) &
        (source_df["speaker_role"] == speaker_role)
    ].copy()

    if len(candidate_df) < target_quota:
        raise ValueError(
            f"Недостаточно строк для {dataset_name} / {speaker_role}: "
            f"available={len(candidate_df)}, target={target_quota}"
        )

    sampled_df = candidate_df.sample(
        n=target_quota,
        random_state=RANDOM_STATE
    ).copy()

    sampled_parts.append(sampled_df)

labeling_candidates_df = pd.concat(sampled_parts, axis=0, ignore_index=True)

print("labeling_candidates_df:", labeling_candidates_df.shape)
display(
    labeling_candidates_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

display(
    labeling_candidates_df
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "count"], ascending=[True, False])
)

labeling_candidates_df: (900, 21)


,dataset_name,count
0,teacher_student_dialogues_ru,600
1,student_feedback_ru,300


,dataset_name,speaker_role,count
0,student_feedback_ru,student,300
1,teacher_student_dialogues_ru,student,300
2,teacher_student_dialogues_ru,teacher_analog,300


In [420]:
# Перемешаем итоговый батч, чтобы строки не шли блоками по источнику

labeling_candidates_df = labeling_candidates_df.sample(
    frac=1.0,
    random_state=RANDOM_STATE
).reset_index(drop=True)

labeling_candidates_df["label_item_id"] = [
    f"LBL_{i:05d}" for i in range(1, len(labeling_candidates_df) + 1)
]

labeling_candidates_df["batch_name"] = BATCH_NAME
labeling_candidates_df["manual_label"] = ""
labeling_candidates_df["labeler_notes"] = ""
labeling_candidates_df["reviewed_flag"] = False
labeling_candidates_df["adjudicated_label"] = ""

print(labeling_candidates_df.shape)
display(labeling_candidates_df.head(10))

(900, 27)


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json,char_len,word_len,text_norm_for_dup,is_duplicate_norm,is_empty_text,is_too_short_words,is_too_short_chars,label_item_id,batch_name,manual_label,labeler_notes,reviewed_flag,adjudicated_label
0,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,7075.0,0.0,student,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...","Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'"", ""original_theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'""}",220,35,"привет! я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",False,False,False,False,LBL_00001,labeling_batch_v2_domain_ru,,,False,
1,student_feedback_ru,domain_education,domain,train.csv,train,NaN,NaN,student,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...","Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 1, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т...",697,106,"электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. выполнение заданий выполняется по методичке, что очень удобно. если будет что-то не понятно, то нат...",False,False,False,False,LBL_00002,labeling_batch_v2_domain_ru,,,False,
2,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,417.0,0.0,student,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...","Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Прототипическая семантика машина"", ""original_theme"": ""Прототипическая семантика машина""}",200,30,"интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",False,False,False,False,LBL_00003,labeling_batch_v2_domain_ru,,,False,
3,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,2803.0,1.0,teacher_analog,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...","Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Тренды моды в описании социальных реформ"", ""original_theme"": ""Тренды

## Формируем master и sheet для ручной разметки

Сохраняем два представления:
- `master` — полный рабочий набор с метаданными;
- `sheet` — упрощённый файл для разметки руками.

In [421]:
master_cols = [
    "label_item_id",
    "batch_name",
    "dataset_name",
    "source_file",
    "split",
    "conversation_id",
    "turn_id",
    "speaker_role",
    "text_clean",
    "raw_label",
    "target_label",
    "label_type",
    "meta_json",
    "char_len",
    "word_len",
    "manual_label",
    "labeler_notes",
    "reviewed_flag",
    "adjudicated_label",
]

existing_master_cols = [c for c in master_cols if c in labeling_candidates_df.columns]
labeling_batch_master_df = labeling_candidates_df[existing_master_cols].copy()

labeling_batch_master_df = labeling_batch_master_df.rename(columns={
    "text_clean": "text"
})

sheet_cols = [
    "label_item_id",
    "text",
    "manual_label",
    "labeler_notes",
]

labeling_batch_sheet_df = labeling_batch_master_df[sheet_cols].copy()

display(labeling_batch_master_df.head(5))
display(labeling_batch_sheet_df.head(5))

,label_item_id,batch_name,dataset_name,source_file,split,conversation_id,turn_id,speaker_role,text,raw_label,target_label,label_type,meta_json,char_len,word_len,manual_label,labeler_notes,reviewed_flag,adjudicated_label
0,LBL_00001,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru,teacher_student_dialogues_ru_turn_level.csv,unknown,7075.0,0.0,student,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'"", ""original_theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'""}",220,35,,,False,
1,LBL_00002,labeling_batch_v2_domain_ru,student_feedback_ru,train.csv,train,NaN,NaN,student,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 1, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т...",697,106,,,False,
2,LBL_00003,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru,teacher_student_dialogues_ru_turn_level.csv,unknown,417.0,0.0,student,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Прототипическая семантика машина"", ""original_theme"": ""Прототипическая семантика машина""}",200,30,,,False,
3,LBL_00004,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru,teacher_student_dialogues_ru_turn_level.csv,unknown,2803.0,1.0,teacher_analog,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Тренды моды в описании социальных реформ"", ""original_theme"": ""Тренды моды в описании социальные реформы""}",489,64,,,False,
4,LBL_00005,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru,teacher_student_dialogues_ru_turn_level.csv,unknown,7283.0,0.0,student,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Ищем лучший способ перевести диалог о 'безопасность и комфорт'"", ""original_theme"": ""Ищем лучший способ перевести диалог о 'безопасность и комфорт'""}",218,31,,,False,


,label_item_id,text,manual_label,labeler_notes
0,LBL_00001,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",,
1,LBL_00002,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",,
2,LBL_00003,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",,
3,LBL_00004,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",,
4,LBL_00005,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо...",,


In [422]:
print("master shape:", labeling_batch_master_df.shape)
print("sheet shape :", labeling_batch_sheet_df.shape)

print("\nРаспределение по источникам:")
display(
    labeling_batch_master_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

print("\nРаспределение по ролям:")
display(
    labeling_batch_master_df["speaker_role"]
    .fillna("<NA>")
    .value_counts(dropna=False)
    .rename_axis("speaker_role")
    .reset_index(name="count")
)

master shape: (900, 19)
sheet shape : (900, 4)

Распределение по источникам:


,dataset_name,count
0,teacher_student_dialogues_ru,600
1,student_feedback_ru,300



Распределение по ролям:


,speaker_role,count
0,student,600
1,teacher_analog,300


## Что проверить сейчас

После сборки `labeling_candidates_df` нужно посмотреть:
1. получилось ли ровно 900 строк;
2. совпадают ли квоты по группам;
3. нет ли пустых текстов;
4. нет ли повторов внутри итогового батча;
5. выглядят ли примеры содержательно разнообразными.

In [423]:
labeling_qc_df = pd.DataFrame([
    {
        "rows_total": len(labeling_batch_master_df),
        "empty_text_rows": int(labeling_batch_master_df["text"].isna().sum()),
        "duplicate_text_rows": int(
            labeling_batch_master_df["text"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .duplicated(keep=False)
            .sum()
        ),
        "unique_conversations": int(labeling_batch_master_df["conversation_id"].nunique(dropna=True)),
    }
])

display(labeling_qc_df)

,rows_total,empty_text_rows,duplicate_text_rows,unique_conversations
0,900,0,0,408


In [424]:
print("Примеры итогового батча:")
display(
    labeling_batch_master_df[
        ["label_item_id", "dataset_name", "speaker_role", "text"]
    ].head(20)
)

Примеры итогового батча:


,label_item_id,dataset_name,speaker_role,text
0,LBL_00001,teacher_student_dialogues_ru,student,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к..."
1,LBL_00002,student_feedback_ru,student,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат..."
2,LBL_00003,teacher_student_dialogues_ru,student,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш..."
3,LBL_00004,teacher_student_dialogues_ru,teacher_analog,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr..."
4,LBL_00005,teacher_student_dialogues_ru,student,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо..."
5,LBL_00006,student_feedback_ru,student,"Электив, знакомящий с базами пользования 1С. Преподаватель отзывчив, занятия интересны. Занятия записываются, что полезно."
6,LBL_00007,teacher_student_dialogues_ru,student,"Интересно, как в русском и английском языках выражают идею 'мгновения' – того самого краткого момента времени? Я заметил, что в английском часто говорят 'in a moment', но в русском это звучит по-д..."
7,LBL_00008,teacher_student_dialogues_ru,student,"Привет! Я перевожу статьи по психологии и часто сталкиваюсь с терминами вроде 'long-term memory' и 'short-term memory'. Как правильно перевести 'долговременное' и 'кратковременное', чтобы не ошиби..."
8,LBL_00009,teacher_student_dialogues_ru,student,"Я слышал, что микрогрин — это такая модная тема в академических кругах. Но как правильно обсуждать это на русском, чтобы не путаться с английскими терминами? Например, 'microgreens' — это просто '..."
9,LBL_00010,teacher_student_dialogues_ru,student,"Привет! Я новичок в стриминге, и меня интересует звуковой дизайн. Как это работает на практике? В английском говорят 'sound design', а по-русски просто 'звуки' или как?"


## Дополняем батч служебными признаками

- добавляем `quota_group`, чтобы потом можно было анализировать разметку по квотируемым группам;
- пересобираем `master` и `sheet` с учётом этого признака;
- сохраняем внутренний audit-слой батча для последующего контроля качества.

In [425]:
labeling_candidates_df["quota_group"] = (
    labeling_candidates_df["dataset_name"].fillna("") +
    " / " +
    labeling_candidates_df["speaker_role"].fillna("")
)

display(
    labeling_candidates_df["quota_group"]
    .value_counts(dropna=False)
    .rename_axis("quota_group")
    .reset_index(name="count")
)

,quota_group,count
0,teacher_student_dialogues_ru / student,300
1,student_feedback_ru / student,300
2,teacher_student_dialogues_ru / teacher_analog,300


In [426]:
master_cols = [
    "label_item_id",
    "batch_name",
    "quota_group",
    "dataset_name",
    "source_group",
    "priority",
    "source_file",
    "split",
    "conversation_id",
    "turn_id",
    "speaker_role",
    "text_clean",
    "raw_label",
    "target_label",
    "label_type",
    "meta_json",
    "char_len",
    "word_len",
    "manual_label",
    "labeler_notes",
    "reviewed_flag",
    "adjudicated_label",
]

existing_master_cols = [c for c in master_cols if c in labeling_candidates_df.columns]

labeling_batch_master_df = labeling_candidates_df[existing_master_cols].copy()
labeling_batch_master_df = labeling_batch_master_df.rename(columns={"text_clean": "text"})

sheet_cols = [
    "label_item_id",
    "text",
    "manual_label",
    "labeler_notes",
]

labeling_batch_sheet_df = labeling_batch_master_df[sheet_cols].copy()

display(labeling_batch_master_df.head(5))
display(labeling_batch_sheet_df.head(5))

,label_item_id,batch_name,quota_group,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text,raw_label,target_label,label_type,meta_json,char_len,word_len,manual_label,labeler_notes,reviewed_flag,adjudicated_label
0,LBL_00001,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,7075.0,0.0,student,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'"", ""original_theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'""}",220,35,,,False,
1,LBL_00002,labeling_batch_v2_domain_ru,student_feedback_ru / student,student_feedback_ru,domain_education,domain,train.csv,train,NaN,NaN,student,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 1, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т...",697,106,,,False,
2,LBL_00003,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,417.0,0.0,student,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Прототипическая семантика машина"", ""original_theme"": ""Прототипическая семантика машина""}",200,30,,,False,
3,LBL_00004,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / teacher_analog,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,2803.0,1.0,teacher_analog,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Тренды моды в описании социальных реформ"", ""original_theme"": ""Тренды моды в описании социальные реформы""}",489,64,,,False,
4,LBL_00005,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,7283.0,0.0,student,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Ищем лучший способ перевести диалог о 'безопасность и комфорт'"", ""original_theme"": ""Ищем лучший способ перевести диалог о 'безопасность и комфорт'""}",218,31,,,False,


,label_item_id,text,manual_label,labeler_notes
0,LBL_00001,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",,
1,LBL_00002,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",,
2,LBL_00003,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",,
3,LBL_00004,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",,
4,LBL_00005,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо...",,


In [427]:
labeling_batch_internal_audit_df = labeling_candidates_df[
    [
        "label_item_id",
        "batch_name",
        "quota_group",
        "dataset_name",
        "speaker_role",
        "conversation_id",
        "turn_id",
        "char_len",
        "word_len",
        "text_norm_for_dup",
        "is_duplicate_norm",
        "is_empty_text",
        "is_too_short_words",
        "is_too_short_chars",
    ]
].copy()

display(labeling_batch_internal_audit_df.head(5))

,label_item_id,batch_name,quota_group,dataset_name,speaker_role,conversation_id,turn_id,char_len,word_len,text_norm_for_dup,is_duplicate_norm,is_empty_text,is_too_short_words,is_too_short_chars
0,LBL_00001,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,student,7075.0,0.0,220,35,"привет! я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",False,False,False,False
1,LBL_00002,labeling_batch_v2_domain_ru,student_feedback_ru / student,student_feedback_ru,student,NaN,NaN,697,106,"электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. выполнение заданий выполняется по методичке, что очень удобно. если будет что-то не понятно, то нат...",False,False,False,False
2,LBL_00003,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,student,417.0,0.0,200,30,"интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",False,False,False,False
3,LBL_00004,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / teacher_analog,teacher_student_dialogues_ru,teacher_analog,2803.0,1.0,489,64,"конечно, давай разберёмся. в моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. на английском это может быть 'gender-neutral fashion trends pr...",False,False,False,False
4,LBL_00005,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,student,7283.0,0.0,218,31,"привет! я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. но мне кажется, что прямой перевод звучит неестественно на русском. как найти лучший спо...",False,False,False,False


## Папки сохранения батча

Сохраняем результат этапа в `03_labeling/batches/<batch_name>/`.
Отдельно пишем:
- `master`
- `sheet`
- summary-таблицы
- metadata / audit

In [428]:
LABELING_DIR = DRIVE_PROJECT_ROOT / "03_labeling"
BATCHES_DIR = LABELING_DIR / "batches"
BATCH_DIR = BATCHES_DIR / BATCH_NAME
METADATA_BATCH_DIR = BATCH_DIR / "metadata"

for folder in [LABELING_DIR, BATCHES_DIR, BATCH_DIR, METADATA_BATCH_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("LABELING_DIR   :", LABELING_DIR)
print("BATCH_DIR      :", BATCH_DIR)
print("METADATA_DIR   :", METADATA_BATCH_DIR)

LABELING_DIR   : /content/drive/MyDrive/tutors_sentiment_project/03_labeling
BATCH_DIR      : /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru
METADATA_DIR   : /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata


In [429]:
batch_source_summary_df = (
    labeling_batch_master_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

batch_role_summary_df = (
    labeling_batch_master_df["speaker_role"]
    .fillna("<NA>")
    .value_counts(dropna=False)
    .rename_axis("speaker_role")
    .reset_index(name="count")
)

batch_quota_summary_df = (
    labeling_batch_master_df
    .groupby(["quota_group", "dataset_name", "speaker_role"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "speaker_role"])
)

batch_qc_summary_df = pd.DataFrame([
    {
        "batch_name": BATCH_NAME,
        "rows_total": len(labeling_batch_master_df),
        "empty_text_rows": int(labeling_batch_master_df["text"].isna().sum()),
        "duplicate_text_rows": int(
            labeling_batch_master_df["text"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .duplicated(keep=False)
            .sum()
        ),
        "unique_conversations": int(labeling_batch_master_df["conversation_id"].nunique(dropna=True)),
        "num_dataset_sources": int(labeling_batch_master_df["dataset_name"].nunique()),
        "num_quota_groups": int(labeling_batch_master_df["quota_group"].nunique()),
    }
])

display(batch_source_summary_df)
display(batch_role_summary_df)
display(batch_quota_summary_df)
display(batch_qc_summary_df)

,dataset_name,count
0,teacher_student_dialogues_ru,600
1,student_feedback_ru,300


,speaker_role,count
0,student,600
1,teacher_analog,300


,quota_group,dataset_name,speaker_role,count
0,student_feedback_ru / student,student_feedback_ru,student,300
1,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,student,300
2,teacher_student_dialogues_ru / teacher_analog,teacher_student_dialogues_ru,teacher_analog,300


,batch_name,rows_total,empty_text_rows,duplicate_text_rows,unique_conversations,num_dataset_sources,num_quota_groups
0,labeling_batch_v2_domain_ru,900,0,0,408,2,3


In [430]:
batch_metadata = {
    "batch_name": BATCH_NAME,
    "random_state": RANDOM_STATE,
    "rows_total": int(len(labeling_batch_master_df)),
    "dataset_sources": batch_source_summary_df.to_dict(orient="records"),
    "speaker_roles": batch_role_summary_df.to_dict(orient="records"),
    "quota_groups": batch_quota_summary_df.to_dict(orient="records"),
    "qc_summary": batch_qc_summary_df.to_dict(orient="records"),
    "notes": {
        "selection_strategy": "manual_quota_sampling",
        "dialogue_constraint": "one_row_per_conversation_within_each_speaker_role_for_teacher_student_dialogues_ru",
        "included_groups": [
            "teacher_student_dialogues_ru / student",
            "teacher_student_dialogues_ru / teacher_analog",
            "student_feedback_ru / student",
        ],
    },
}

## Сохраняем артефакты батча

In [431]:
master_csv_path = BATCH_DIR / f"{BATCH_NAME}_master.csv"
master_parquet_path = BATCH_DIR / f"{BATCH_NAME}_master.parquet"

sheet_csv_path = BATCH_DIR / f"{BATCH_NAME}_sheet.csv"
sheet_parquet_path = BATCH_DIR / f"{BATCH_NAME}_sheet.parquet"

source_summary_csv_path = METADATA_BATCH_DIR / f"{BATCH_NAME}_source_summary.csv"
role_summary_csv_path = METADATA_BATCH_DIR / f"{BATCH_NAME}_role_summary.csv"
quota_summary_csv_path = METADATA_BATCH_DIR / f"{BATCH_NAME}_quota_summary.csv"
qc_summary_csv_path = METADATA_BATCH_DIR / f"{BATCH_NAME}_qc_summary.csv"
internal_audit_csv_path = METADATA_BATCH_DIR / f"{BATCH_NAME}_internal_audit.csv"
metadata_json_path = METADATA_BATCH_DIR / f"{BATCH_NAME}_metadata.json"

labeling_batch_master_df.to_csv(master_csv_path, index=False)
labeling_batch_master_df.to_parquet(master_parquet_path, index=False)

labeling_batch_sheet_df.to_csv(sheet_csv_path, index=False)
labeling_batch_sheet_df.to_parquet(sheet_parquet_path, index=False)

batch_source_summary_df.to_csv(source_summary_csv_path, index=False)
batch_role_summary_df.to_csv(role_summary_csv_path, index=False)
batch_quota_summary_df.to_csv(quota_summary_csv_path, index=False)
batch_qc_summary_df.to_csv(qc_summary_csv_path, index=False)
labeling_batch_internal_audit_df.to_csv(internal_audit_csv_path, index=False)

with open(metadata_json_path, "w", encoding="utf-8") as f:
    json.dump(batch_metadata, f, ensure_ascii=False, indent=2)

print("Сохранено:")
print(" -", master_csv_path)
print(" -", master_parquet_path)
print(" -", sheet_csv_path)
print(" -", sheet_parquet_path)
print(" -", source_summary_csv_path)
print(" -", role_summary_csv_path)
print(" -", quota_summary_csv_path)
print(" -", qc_summary_csv_path)
print(" -", internal_audit_csv_path)
print(" -", metadata_json_path)

Сохранено:
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_master.csv
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_master.parquet
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_sheet.csv
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_sheet.parquet
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_source_summary.csv
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_role_summary.csv
 - /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_

In [432]:
saved_files_df = pd.DataFrame([
    {"artifact": "master_csv", "path": str(master_csv_path), "exists": master_csv_path.exists()},
    {"artifact": "master_parquet", "path": str(master_parquet_path), "exists": master_parquet_path.exists()},
    {"artifact": "sheet_csv", "path": str(sheet_csv_path), "exists": sheet_csv_path.exists()},
    {"artifact": "sheet_parquet", "path": str(sheet_parquet_path), "exists": sheet_parquet_path.exists()},
    {"artifact": "source_summary_csv", "path": str(source_summary_csv_path), "exists": source_summary_csv_path.exists()},
    {"artifact": "role_summary_csv", "path": str(role_summary_csv_path), "exists": role_summary_csv_path.exists()},
    {"artifact": "quota_summary_csv", "path": str(quota_summary_csv_path), "exists": quota_summary_csv_path.exists()},
    {"artifact": "qc_summary_csv", "path": str(qc_summary_csv_path), "exists": qc_summary_csv_path.exists()},
    {"artifact": "internal_audit_csv", "path": str(internal_audit_csv_path), "exists": internal_audit_csv_path.exists()},
    {"artifact": "metadata_json", "path": str(metadata_json_path), "exists": metadata_json_path.exists()},
])

display(saved_files_df)

,artifact,path,exists
0,master_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_master.csv,True
1,master_parquet,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_master.parquet,True
2,sheet_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_sheet.csv,True
3,sheet_parquet,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_sheet.parquet,True
4,source_summary_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_source_summary.csv,True
5,role_summary_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_role_summary.csv,True
6,quota_summary_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_quota_summary.csv,True
7,qc_summary_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_qc_summary.csv,True
8,internal_audit_csv,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_internal_audit.csv,True
9,metadata_json,/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/metadata/labeling_batch_v2_domain_ru_metadata.json,True


## Итог промежуточного этапа `03_build_target_dataset`

На этом шаге мы:
- взяли cleaned доменный пул;
- отфильтровали короткие и повторяющиеся тексты;
- собрали квотированный батч на ручную разметку;
- сформировали `master` и `sheet`;
- сохранили metadata и QC-артефакты.

следующие шаги:
- загрузка обратно размеченного `sheet`;
- объединение ручных меток с `master`;
- построение доменного gold-label слоя;
- объединение его с уже размеченным `sentiment_core`;
- формирование итоговых `train / validation / test`;
- сохранение финальных датасетов

In [433]:
final_stage_summary = {
    "batch_name": BATCH_NAME,
    "rows_master": int(len(labeling_batch_master_df)),
    "rows_sheet": int(len(labeling_batch_sheet_df)),
    "source_counts": batch_source_summary_df.to_dict(orient="records"),
    "role_counts": batch_role_summary_df.to_dict(orient="records"),
    "qc": batch_qc_summary_df.to_dict(orient="records"),
    "saved_to": str(BATCH_DIR),
}

final_stage_summary

{'batch_name': 'labeling_batch_v2_domain_ru',
 'rows_master': 900,
 'rows_sheet': 900,
 'source_counts': [{'dataset_name': 'teacher_student_dialogues_ru',
   'count': 600},
  {'dataset_name': 'student_feedback_ru', 'count': 300}],
 'role_counts': [{'speaker_role': 'student', 'count': 600},
  {'speaker_role': 'teacher_analog', 'count': 300}],
 'qc': [{'batch_name': 'labeling_batch_v2_domain_ru',
   'rows_total': 900,
   'empty_text_rows': 0,
   'duplicate_text_rows': 0,
   'unique_conversations': 408,
   'num_dataset_sources': 2,
   'num_quota_groups': 3}],
 'saved_to': '/content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru'}

## Переходим ко второй половине `03_build_target_dataset`

Мы должны:
- загрузить обратно размеченный `sheet`;
- проверить корректность заполненных меток;
- объединить размеченный `sheet` с `master`;
- получить доменный gold-label слой;
- подготовить его для объединения с основным sentiment-ядром.

In [434]:
ALLOWED_MANUAL_LABELS = {"positive", "neutral", "negative"}

annotated_sheet_candidates = [
    BATCH_DIR / f"{BATCH_NAME}_sheet_labeled.csv",
    BATCH_DIR / f"{BATCH_NAME}_sheet_annotated.csv",
    BATCH_DIR / f"{BATCH_NAME}_sheet_filled.csv",
]

annotated_sheet_path = None
for path in annotated_sheet_candidates:
    if path.exists():
        annotated_sheet_path = path
        break

print("annotated_sheet_path:", annotated_sheet_path)

annotated_sheet_path: /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches/labeling_batch_v2_domain_ru/labeling_batch_v2_domain_ru_sheet_labeled.csv


In [435]:
if annotated_sheet_path is None:
    raise FileNotFoundError(
        "Размеченный sheet пока не найден. "
        "Сначала заполни manual_label и сохрани файл обратно в папку батча."
    )

annotated_sheet_df = pd.read_csv(annotated_sheet_path)
print("annotated_sheet_df shape:", annotated_sheet_df.shape)
display(annotated_sheet_df.head(10))

annotated_sheet_df shape: (900, 4)


,label_item_id,text,manual_label,labeler_notes
0,LBL_00001,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",negative,NaN
1,LBL_00002,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",positive,NaN
2,LBL_00003,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",neutral,NaN
3,LBL_00004,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",positive,NaN
4,LBL_00005,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо...",negative,NaN
5,LBL_00006,"Электив, знакомящий с базами пользования 1С. Преподаватель отзывчив, занятия интересны. Занятия записываются, что полезно.",positive,NaN
6,LBL_00007,"Интересно, как в русском и английском языках выражают идею 'мгновения' – того самого краткого момента времени? Я заметил, что в английском часто говорят 'in a moment', но в русском это звучит по-д...",neutral,NaN
7,LBL_00008,"Привет! Я перевожу статьи по психологии и часто сталкиваюсь с терминами вроде 'long-term memory' и 'short-term memory'. Как правильно перевести 'долговременное' и 'кратковременное', чтобы не ошиби...",negative,NaN
8,LBL_00009,"Я слышал, что микрогрин — это такая модная тема в академических кругах. Но как правильно обсуждать это на русском, чтобы не путаться с английскими терминами? Например, 'microgreens' — это просто '...",neutral,NaN
9,LBL_00010,"Привет! Я новичок в стриминге, и меня интересует звуковой дизайн. Как это работает на практике? В английском говорят 'sound design', а по-русски просто 'звуки' или как?",neutral,NaN


In [436]:
required_sheet_cols = {"label_item_id", "text", "manual_label"}
missing_sheet_cols = required_sheet_cols - set(annotated_sheet_df.columns)

if missing_sheet_cols:
    raise ValueError(f"В размеченном sheet не хватает колонок: {missing_sheet_cols}")

annotated_sheet_df["manual_label"] = (
    annotated_sheet_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

display(
    annotated_sheet_df["manual_label"]
    .value_counts(dropna=False)
    .rename_axis("manual_label")
    .reset_index(name="count")
)

,manual_label,count
0,positive,426
1,neutral,388
2,negative,86


In [437]:
invalid_labels_df = annotated_sheet_df.loc[
    (annotated_sheet_df["manual_label"] != "") &
    (~annotated_sheet_df["manual_label"].isin(ALLOWED_MANUAL_LABELS))
].copy()

print("invalid_labels_df:", invalid_labels_df.shape)
display(invalid_labels_df.head(20))

invalid_labels_df: (0, 4)


,label_item_id,text,manual_label,labeler_notes


In [438]:
filled_labels_df = annotated_sheet_df.loc[
    annotated_sheet_df["manual_label"].isin(ALLOWED_MANUAL_LABELS)
].copy()

print("filled_labels_df shape:", filled_labels_df.shape)

display(
    filled_labels_df["manual_label"]
    .value_counts(dropna=False)
    .rename_axis("manual_label")
    .reset_index(name="count")
)

filled_labels_df shape: (900, 4)


,manual_label,count
0,positive,426
1,neutral,388
2,negative,86


In [439]:
if filled_labels_df.empty:
    raise ValueError(
        "В размеченном sheet не найдено ни одной заполненной метки manual_label. "
        "Сейчас загружен файл без ручной разметки или разметка ещё не внесена."
    )

In [440]:
domain_labeled_df = labeling_batch_master_df.merge(
    filled_labels_df[["label_item_id", "manual_label"]],
    on="label_item_id",
    how="left",
    suffixes=("", "_from_sheet")
)

if "manual_label_from_sheet" in domain_labeled_df.columns:
    domain_labeled_df["manual_label"] = domain_labeled_df["manual_label_from_sheet"].fillna(domain_labeled_df["manual_label"])
    domain_labeled_df = domain_labeled_df.drop(columns=["manual_label_from_sheet"])

print("domain_labeled_df shape:", domain_labeled_df.shape)
display(domain_labeled_df.head(10))

domain_labeled_df shape: (900, 22)


,label_item_id,batch_name,quota_group,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text,raw_label,target_label,label_type,meta_json,char_len,word_len,manual_label,labeler_notes,reviewed_flag,adjudicated_label
0,LBL_00001,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,7075.0,0.0,student,"Привет! Я пытаюсь переводить статьи о домашних и уличных животных с английского на русский, но часто путаюсь в терминах. Например, как правильно перевести 'stray cat' – это же не просто 'уличная к...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'"", ""original_theme"": ""Как избежать ошибок при переводе текстов о 'домашнее и уличное'""}",220,35,negative,,False,
1,LBL_00002,labeling_batch_v2_domain_ru,student_feedback_ru / student,student_feedback_ru,domain_education,domain,train.csv,train,NaN,NaN,student,"Электив подходит для всех направлений, преподается все с нуля, следовательно легко будет понять. Выполнение заданий выполняется по методичке, что очень удобно. Если будет что-то не понятно, то Нат...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 1, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т...",697,106,positive,,False,
2,LBL_00003,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,417.0,0.0,student,"Интересно, что такое прототипическая семантика, и как это относится к слову 'машина' в русском языке? Я заметил, что в английском 'machine' кажется более узким, а в русском 'машина' используется ш...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Прототипическая семантика машина"", ""original_theme"": ""Прототипическая семантика машина""}",200,30,neutral,,False,
3,LBL_00004,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / teacher_analog,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,2803.0,1.0,teacher_analog,"Конечно, давай разберёмся. В моде тренды вроде гендерно-нейтральной одежды или устойчивой моды часто связаны с социальными реформами. На английском это может быть 'gender-neutral fashion trends pr...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Тренды моды в описании социальных реформ"", ""original_theme"": ""Тренды моды в описании социальные реформы""}",489,64,positive,,False,
4,LBL_00005,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dialogues_ru,domain_dialogue_core,domain_core,teacher_student_dialogues_ru_turn_level.csv,unknown,7283.0,0.0,student,"Привет! Я работаю над переводом английского диалога, где обсуждается безопасность и комфорт в путешествиях. Но мне кажется, что прямой перевод звучит неестественно на русском. Как найти лучший спо...",NaN,NaN,unlabeled_dialogue,"{""theme"": ""Ищем лучший способ перевести диалог о 'безопасность и комфорт'"", ""original_theme"": ""Ищем лучший способ перевести диалог о 'безопасность и комфорт'""}",218,31,negative,,False,
5,LBL_00006,labeling_batch_v2_domain_ru,student_feedback_ru / student,student_feedback_ru,domain_education,domain,validation.csv,validation,NaN,NaN,student,"Электив, знакомящий с базами пользования 1С. Преподаватель отзывчив, занятия интересны. Занятия записываются, что полезно.",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т...",122,14,positive,,False,
6,LBL_00007,labeling_batch_v2_domain_ru,teacher_student_dialogues_ru / student,teacher_student_dial

In [441]:
domain_gold_df = domain_labeled_df.loc[
    domain_labeled_df["manual_label"].isin(ALLOWED_MANUAL_LABELS)
].copy()

domain_gold_df["target_label"] = domain_gold_df["manual_label"]
domain_gold_df["label_source"] = "manual_domain_labeling"
domain_gold_df["is_domain_labeled"] = True

print("domain_gold_df shape:", domain_gold_df.shape)

display(
    domain_gold_df["target_label"]
    .value_counts(dropna=False)
    .rename_axis("target_label")
    .reset_index(name="count")
)

display(
    domain_gold_df.groupby(["dataset_name", "speaker_role", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "speaker_role", "count"], ascending=[True, True, False])
)

domain_gold_df shape: (900, 24)


,target_label,count
0,positive,426
1,neutral,388
2,negative,86


,dataset_name,speaker_role,target_label,count
2,student_feedback_ru,student,positive,201
1,student_feedback_ru,student,neutral,80
0,student_feedback_ru,student,negative,19
4,teacher_student_dialogues_ru,student,neutral,236
3,teacher_student_dialogues_ru,student,negative,64
7,teacher_student_dialogues_ru,teacher_analog,positive,225
6,teacher_student_dialogues_ru,teacher_analog,neutral,72
5,teacher_student_dialogues_ru,teacher_analog,negative,3


In [442]:
display(
    domain_gold_df
    .groupby(["quota_group", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["quota_group", "count"], ascending=[True, False])
)

,quota_group,target_label,count
2,student_feedback_ru / student,positive,201
1,student_feedback_ru / student,neutral,80
0,student_feedback_ru / student,negative,19
4,teacher_student_dialogues_ru / student,neutral,236
3,teacher_student_dialogues_ru / student,negative,64
7,teacher_student_dialogues_ru / teacher_analog,positive,225
6,teacher_student_dialogues_ru / teacher_analog,neutral,72
5,teacher_student_dialogues_ru / teacher_analog,negative,3


## Переходим к финальному supervised dataset

На этом шаге:
- приводим `sentiment_core_prepared` к финальной форме;
- строим сплиты для `domain_gold_df`;
- объединяем размеченный доменный слой с существующим sentiment-ядром;
- формируем итоговые `train / validation / test` наборы.

In [443]:
from sklearn.model_selection import train_test_split

## Вспомогательные функции для безопасного разбиения

Используем "безопасное" разбиение:
- если классов или строк слишком мало для стратификации, делаем обычный split;
- если возможно, используем stratify.

In [444]:
def safe_row_split(df, test_size, stratify_col="target_label", random_state=42):
    if df.empty:
        return df.copy(), df.iloc[0:0].copy()

    if len(df) < 2:
        return df.copy(), df.iloc[0:0].copy()

    use_stratify = False
    if stratify_col in df.columns:
        value_counts = df[stratify_col].value_counts(dropna=False)
        if len(value_counts) >= 2 and value_counts.min() >= 2:
            use_stratify = True

    if use_stratify:
        train_df, test_df = train_test_split(
            df,
            test_size=test_size,
            random_state=random_state,
            stratify=df[stratify_col]
        )
    else:
        train_df, test_df = train_test_split(
            df,
            test_size=test_size,
            random_state=random_state
        )

    return train_df.copy(), test_df.copy()


def safe_group_split(df, group_col, label_col="target_label", test_size=0.2, random_state=42):
    if df.empty:
        return df.copy(), df.iloc[0:0].copy()

    if group_col not in df.columns:
        return safe_row_split(df, test_size=test_size, stratify_col=label_col, random_state=random_state)

    group_df = (
        df.groupby(group_col)[label_col]
        .agg(lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0])
        .reset_index(name="group_target_label")
    )

    if len(group_df) < 2:
        return df.copy(), df.iloc[0:0].copy()

    use_stratify = False
    value_counts = group_df["group_target_label"].value_counts(dropna=False)
    if len(value_counts) >= 2 and value_counts.min() >= 2:
        use_stratify = True

    if use_stratify:
        train_groups, test_groups = train_test_split(
            group_df[group_col],
            test_size=test_size,
            random_state=random_state,
            stratify=group_df["group_target_label"]
        )
    else:
        train_groups, test_groups = train_test_split(
            group_df[group_col],
            test_size=test_size,
            random_state=random_state
        )

    train_df = df[df[group_col].isin(train_groups)].copy()
    test_df = df[df[group_col].isin(test_groups)].copy()

    return train_df, test_df

## Готовим `sentiment_core_final_df`

Для sentiment-ядра:
- нормализуем значения `split`;
- существующие `train / val / test` сохраняем;
- строки с `unknown` дополнительно делим на `train / val`;
- источник метки помечаем как `existing_sentiment_core`.

In [445]:
sentiment_core_final_df = sentiment_core_df.copy()

sentiment_core_final_df["split"] = (
    sentiment_core_final_df["split"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
    .str.lower()
)

split_map = {
    "validation": "val",
    "valid": "val",
    "dev": "val",
    "train": "train",
    "val": "val",
    "test": "test",
    "unknown": "unknown",
}
sentiment_core_final_df["split"] = sentiment_core_final_df["split"].map(lambda x: split_map.get(x, x))

sentiment_core_final_df["label_source"] = "existing_sentiment_core"
sentiment_core_final_df["is_domain_labeled"] = False

print("sentiment_core_final_df shape:", sentiment_core_final_df.shape)
display(
    sentiment_core_final_df["split"]
    .value_counts(dropna=False)
    .rename_axis("split")
    .reset_index(name="count")
)

sentiment_core_final_df shape: (34331, 16)


,split,count
0,unknown,21064
1,train,7971
2,test,4425
3,val,871


In [446]:
sent_known_split_df = sentiment_core_final_df.loc[
    sentiment_core_final_df["split"].isin(["train", "val", "test"])
].copy()

sent_unknown_split_df = sentiment_core_final_df.loc[
    sentiment_core_final_df["split"] == "unknown"
].copy()

print("sent_known_split_df :", sent_known_split_df.shape)
print("sent_unknown_split_df:", sent_unknown_split_df.shape)

display(
    sent_known_split_df
    .groupby(["dataset_name", "split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "split", "count"], ascending=[True, True, False])
)

sent_known_split_df : (13267, 16)
sent_unknown_split_df: (21064, 16)


,dataset_name,split,target_label,count
1,rusentiment,test,neutral,1420
2,rusentiment,test,positive,536
0,rusentiment,test,negative,258
4,rusentitweet,test,neutral,1068
3,rusentitweet,test,negative,660
5,rusentitweet,test,positive,483
7,rusentitweet,train,neutral,3835
6,rusentitweet,train,negative,2393
8,rusentitweet,train,positive,1743
10,rusentitweet,val,neutral,438


In [447]:
sent_unknown_train_df, sent_unknown_val_df = safe_row_split(
    sent_unknown_split_df,
    test_size=0.15,
    stratify_col="target_label",
    random_state=42
)

sent_unknown_train_df["split"] = "train"
sent_unknown_val_df["split"] = "val"

print("sent_unknown_train_df:", sent_unknown_train_df.shape)
print("sent_unknown_val_df  :", sent_unknown_val_df.shape)

display(
    pd.concat([
        sent_unknown_train_df.assign(part="unknown_to_train"),
        sent_unknown_val_df.assign(part="unknown_to_val"),
    ])
    .groupby(["part", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["part", "count"], ascending=[True, False])
)

sent_unknown_train_df: (17904, 16)
sent_unknown_val_df  : (3160, 16)


,part,target_label,count
1,unknown_to_train,neutral,9605
2,unknown_to_train,positive,5193
0,unknown_to_train,negative,3106
4,unknown_to_val,neutral,1695
5,unknown_to_val,positive,917
3,unknown_to_val,negative,548


In [448]:
sentiment_core_final_df = pd.concat(
    [sent_known_split_df, sent_unknown_train_df, sent_unknown_val_df],
    axis=0,
    ignore_index=True
)

print("sentiment_core_final_df:", sentiment_core_final_df.shape)

display(
    sentiment_core_final_df
    .groupby(["split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)

display(
    sentiment_core_final_df
    .groupby(["dataset_name", "split"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "split"])
)

sentiment_core_final_df: (34331, 16)


,split,target_label,count
1,test,neutral,2488
2,test,positive,1019
0,test,negative,918
4,train,neutral,13440
5,train,positive,6936
3,train,negative,5499
7,val,neutral,2133
8,val,positive,1105
6,val,negative,793


,dataset_name,split,count
0,rusentiment,test,2214
1,rusentiment,train,17904
2,rusentiment,val,3160
3,rusentitweet,test,2211
4,rusentitweet,train,7971
5,rusentitweet,val,871


## Строим сплиты для `domain_gold_df`

Для доменного слоя используем разный подход:
- `teacher_student_dialogues_ru` — split по `conversation_id`, чтобы реплики одной беседы не попадали в разные части;
- `student_feedback_ru` — обычный stratified split по строкам.

Целевые пропорции:
- train = 80%
- validation = 10%
- test = 10%

In [449]:
domain_gold_dialogues_df = domain_gold_df.loc[
    domain_gold_df["dataset_name"] == "teacher_student_dialogues_ru"
].copy()

domain_gold_feedback_df = domain_gold_df.loc[
    domain_gold_df["dataset_name"] == "student_feedback_ru"
].copy()

print("domain_gold_dialogues_df:", domain_gold_dialogues_df.shape)
print("domain_gold_feedback_df :", domain_gold_feedback_df.shape)

display(
    domain_gold_df.groupby(["dataset_name", "speaker_role", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "speaker_role", "count"], ascending=[True, True, False])
)

domain_gold_dialogues_df: (600, 24)
domain_gold_feedback_df : (300, 24)


,dataset_name,speaker_role,target_label,count
2,student_feedback_ru,student,positive,201
1,student_feedback_ru,student,neutral,80
0,student_feedback_ru,student,negative,19
4,teacher_student_dialogues_ru,student,neutral,236
3,teacher_student_dialogues_ru,student,negative,64
7,teacher_student_dialogues_ru,teacher_analog,positive,225
6,teacher_student_dialogues_ru,teacher_analog,neutral,72
5,teacher_student_dialogues_ru,teacher_analog,negative,3


In [450]:
dlg_train_df, dlg_temp_df = safe_group_split(
    domain_gold_dialogues_df,
    group_col="conversation_id",
    label_col="target_label",
    test_size=0.20,
    random_state=42
)

dlg_val_df, dlg_test_df = safe_group_split(
    dlg_temp_df,
    group_col="conversation_id",
    label_col="target_label",
    test_size=0.50,
    random_state=43
)

dlg_train_df["split"] = "train"
dlg_val_df["split"] = "val"
dlg_test_df["split"] = "test"

print("dlg_train_df:", dlg_train_df.shape)
print("dlg_val_df  :", dlg_val_df.shape)
print("dlg_test_df :", dlg_test_df.shape)

dlg_train_df: (481, 24)
dlg_val_df  : (59, 24)
dlg_test_df : (60, 24)


In [451]:
fb_train_df, fb_temp_df = safe_row_split(
    domain_gold_feedback_df,
    test_size=0.20,
    stratify_col="target_label",
    random_state=42
)

fb_val_df, fb_test_df = safe_row_split(
    fb_temp_df,
    test_size=0.50,
    stratify_col="target_label",
    random_state=43
)

fb_train_df["split"] = "train"
fb_val_df["split"] = "val"
fb_test_df["split"] = "test"

print("fb_train_df:", fb_train_df.shape)
print("fb_val_df  :", fb_val_df.shape)
print("fb_test_df :", fb_test_df.shape)

fb_train_df: (240, 24)
fb_val_df  : (30, 24)
fb_test_df : (30, 24)


In [452]:
domain_gold_final_df = pd.concat(
    [
        dlg_train_df, dlg_val_df, dlg_test_df,
        fb_train_df, fb_val_df, fb_test_df,
    ],
    axis=0,
    ignore_index=True
)

print("domain_gold_final_df:", domain_gold_final_df.shape)

display(
    domain_gold_final_df
    .groupby(["split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)

display(
    domain_gold_final_df
    .groupby(["dataset_name", "split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "split", "count"], ascending=[True, True, False])
)

domain_gold_final_df: (900, 24)


,split,target_label,count
2,test,positive,42
1,test,neutral,39
0,test,negative,9
5,train,positive,344
4,train,neutral,308
3,train,negative,69
7,val,neutral,41
8,val,positive,40
6,val,negative,8


,dataset_name,split,target_label,count
2,student_feedback_ru,test,positive,20
1,student_feedback_ru,test,neutral,8
0,student_feedback_ru,test,negative,2
5,student_feedback_ru,train,positive,161
4,student_feedback_ru,train,neutral,64
3,student_feedback_ru,train,negative,15
8,student_feedback_ru,val,positive,20
7,student_feedback_ru,val,neutral,8
6,student_feedback_ru,val,negative,2
10,teacher_student_dialogues_ru,test,neutral,31


## Собираем общий supervised dataset

Теперь объединяем:
- `sentiment_core_final_df`
- `domain_gold_final_df`

После этого проверяем:
- общий размер;
- распределение по источникам;
- распределение по сплитам;
- распределение по классам.

In [453]:
supervised_common_cols = sorted(
    set(sentiment_core_final_df.columns).union(set(domain_gold_final_df.columns))
)

for col in supervised_common_cols:
    if col not in sentiment_core_final_df.columns:
        sentiment_core_final_df[col] = np.nan
    if col not in domain_gold_final_df.columns:
        domain_gold_final_df[col] = np.nan

sentiment_core_final_df = sentiment_core_final_df[supervised_common_cols].copy()
domain_gold_final_df = domain_gold_final_df[supervised_common_cols].copy()

final_supervised_df = pd.concat(
    [sentiment_core_final_df, domain_gold_final_df],
    axis=0,
    ignore_index=True
)

print("final_supervised_df:", final_supervised_df.shape)
display(final_supervised_df.head(10))

final_supervised_df: (35231, 26)


,adjudicated_label,batch_name,char_len,conversation_id,dataset_name,is_domain_labeled,label_item_id,label_source,label_type,labeler_notes,manual_label,meta_json,priority,quota_group,raw_label,reviewed_flag,source_file,source_group,speaker_role,split,target_label,text,text_clean,text_raw,turn_id,word_len
0,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,neutral,NaN,rusentiment_test.csv,sentiment_core,NaN,test,neutral,NaN,"Александр, тебе к лицу эта пушка :)","Александр, тебе к лицу эта пушка :)",NaN,NaN
1,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,positive,NaN,rusentiment_test.csv,sentiment_core,NaN,test,positive,NaN,"Скоро ты вернешься домой, грязный, не бритый но такой любимый ❤","Скоро ты вернешься домой, грязный, не бритый но такой любимый ❤",NaN,NaN
2,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,neutral,NaN,rusentiment_test.csv,sentiment_core,NaN,test,neutral,NaN,помниш...)),помниш...)),NaN,NaN
3,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,positive,NaN,rusentiment_test.csv,sentiment_core,NaN,test,positive,NaN,Наши красавцы. 1:3 в первом периоде и 7:3 в итоге.,Наши красавцы. 1:3 в первом периоде и 7:3 в итоге.,NaN,NaN
4,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,neutral,NaN,rusentiment_test.csv,sentiment_core,NaN,test,neutral,NaN,20% усилий приносят 80% результата,20% усилий приносят 80% результата,NaN,NaN
5,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,positive,NaN,rusentiment_test.csv,sentiment_core,NaN,test,positive,NaN,ты для меня самый родной!!!,ты для меня самый родной!!!,NaN,NaN
6,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,negative,NaN,rusentiment_test.csv,sentiment_core,NaN,test,negative,NaN,"нельзя общаться с людьми, которые нуждаются в тебе меньше чем ты в них... это самоунижение","нельзя общаться с людьми, которые нуждаются в тебе меньше чем ты в них... это самоунижение",NaN,NaN
7,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,neutral,NaN,rusentiment_test.csv,sentiment_core,NaN,test,neutral,NaN,мы с Мангаром решили наведаться в гости к тебе),мы с Мангаром решили наведаться в гости к тебе),NaN,NaN
8,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,neutral,NaN,rusentiment_test.csv,sentiment_core,NaN,test,neutral,NaN,"Друзья, скиньте клёвой музычки, в тачке не чего слушать???","Друзья, скиньте клёвой музычки, в тачке не чего слушать???",NaN,NaN
9,NaN,NaN,NaN,NaN,rusentiment,False,NaN,existing_sentiment_core,sentiment_3class_candidate,NaN,NaN,{},core,NaN,positive,NaN,rusentiment_test.csv,sentiment_core,NaN,test,positive,NaN,Селфи дня )),Селфи дня )),NaN,NaN


In [454]:
display(
    final_supervised_df["split"]
    .value_counts(dropna=False)
    .rename_axis("split")
    .reset_index(name="count")
)

display(
    final_supervised_df["target_label"]
    .value_counts(dropna=False)
    .rename_axis("target_label")
    .reset_index(name="count")
)

display(
    final_supervised_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

,split,count
0,train,26596
1,test,4515
2,val,4120


,target_label,count
0,neutral,18449
1,positive,9486
2,negative,7296


,dataset_name,count
0,rusentiment,23278
1,rusentitweet,11053
2,teacher_student_dialogues_ru,600
3,student_feedback_ru,300


In [455]:
display(
    final_supervised_df
    .groupby(["split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)

display(
    final_supervised_df
    .groupby(["dataset_name", "split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "split", "count"], ascending=[True, True, False])
)

,split,target_label,count
1,test,neutral,2527
2,test,positive,1061
0,test,negative,927
4,train,neutral,13748
5,train,positive,7280
3,train,negative,5568
7,val,neutral,2174
8,val,positive,1145
6,val,negative,801


,dataset_name,split,target_label,count
1,rusentiment,test,neutral,1420
2,rusentiment,test,positive,536
0,rusentiment,test,negative,258
4,rusentiment,train,neutral,9605
5,rusentiment,train,positive,5193
3,rusentiment,train,negative,3106
7,rusentiment,val,neutral,1695
8,rusentiment,val,positive,917
6,rusentiment,val,negative,548
10,rusentitweet,test,neutral,1068


In [456]:
train_df = final_supervised_df.loc[final_supervised_df["split"] == "train"].copy()
val_df = final_supervised_df.loc[final_supervised_df["split"] == "val"].copy()
test_df = final_supervised_df.loc[final_supervised_df["split"] == "test"].copy()

print("train_df:", train_df.shape)
print("val_df  :", val_df.shape)
print("test_df :", test_df.shape)

train_df: (26596, 26)
val_df  : (4120, 26)
test_df : (4515, 26)


## Сохраняем итоговые supervised-наборы

На этом шаге сохраняем:
- полный `final_supervised_df`;
- итоговые `train / val / test`;
- компактные версии наборов для обучения;
- summary и metadata в `04_datasets_ready`.

In [457]:
DATASETS_READY_DIR = DRIVE_PROJECT_ROOT / "04_datasets_ready"
TRAIN_DIR = DATASETS_READY_DIR / "train"
VALIDATION_DIR = DATASETS_READY_DIR / "validation"
TEST_DIR = DATASETS_READY_DIR / "test"
METADATA_DIR = DATASETS_READY_DIR / "metadata"

for folder in [DATASETS_READY_DIR, TRAIN_DIR, VALIDATION_DIR, TEST_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATASETS_READY_DIR:", DATASETS_READY_DIR)
print("TRAIN_DIR         :", TRAIN_DIR)
print("VALIDATION_DIR    :", VALIDATION_DIR)
print("TEST_DIR          :", TEST_DIR)
print("METADATA_DIR      :", METADATA_DIR)

DATASETS_READY_DIR: /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready
TRAIN_DIR         : /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train
VALIDATION_DIR    : /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/validation
TEST_DIR          : /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/test
METADATA_DIR      : /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/metadata


In [458]:
# Полные версии (со всеми колонками)
final_supervised_csv_path = DATASETS_READY_DIR / "final_supervised_dataset.csv"
final_supervised_parquet_path = DATASETS_READY_DIR / "final_supervised_dataset.parquet"

train_full_csv_path = TRAIN_DIR / "train_full.csv"
train_full_parquet_path = TRAIN_DIR / "train_full.parquet"

val_full_csv_path = VALIDATION_DIR / "val_full.csv"
val_full_parquet_path = VALIDATION_DIR / "val_full.parquet"

test_full_csv_path = TEST_DIR / "test_full.csv"
test_full_parquet_path = TEST_DIR / "test_full.parquet"

In [459]:
# Компактные версии для обучения модели
compact_cols_priority = [
    "dataset_name",
    "split",
    "text_clean",
    "target_label",
    "label_source",
    "is_domain_labeled",
    "speaker_role",
    "conversation_id",
    "turn_id",
]

compact_cols = [c for c in compact_cols_priority if c in final_supervised_df.columns]

train_compact_df = train_df[compact_cols].copy()
val_compact_df = val_df[compact_cols].copy()
test_compact_df = test_df[compact_cols].copy()

# Для компактных версий переименуем text_clean - text
rename_map = {}
if "text_clean" in compact_cols:
    rename_map["text_clean"] = "text"

train_compact_df = train_compact_df.rename(columns=rename_map)
val_compact_df = val_compact_df.rename(columns=rename_map)
test_compact_df = test_compact_df.rename(columns=rename_map)

train_compact_csv_path = TRAIN_DIR / "train_compact.csv"
train_compact_parquet_path = TRAIN_DIR / "train_compact.parquet"

val_compact_csv_path = VALIDATION_DIR / "val_compact.csv"
val_compact_parquet_path = VALIDATION_DIR / "val_compact.parquet"

test_compact_csv_path = TEST_DIR / "test_compact.csv"
test_compact_parquet_path = TEST_DIR / "test_compact.parquet"

display(train_compact_df.head(5))
display(val_compact_df.head(5))
display(test_compact_df.head(5))

,dataset_name,split,text,target_label,label_source,is_domain_labeled,speaker_role,conversation_id,turn_id
4425,rusentitweet,train,Вообще то нужно потому что я бревно,negative,existing_sentiment_core,False,NaN,NaN,1.262287e+18
4426,rusentitweet,train,сегодня на биологии истерично начала ржать хотя хотелось лишь плакать,negative,existing_sentiment_core,False,NaN,NaN,1.338524e+18
4427,rusentitweet,train,"У меня слов блять нет. Я всю ночь учила хирургию, а в итоге она спросила 4 человека. И спрашивает ВООБЩЕ не по теме",negative,existing_sentiment_core,False,NaN,NaN,1.301334e+18
4428,rusentitweet,train,хоть кто-то так думает✌️😔,positive,existing_sentiment_core,False,NaN,NaN,1.244627e+18
4429,rusentitweet,train,Я сообщу если что😂💛,neutral,existing_sentiment_core,False,NaN,NaN,1.311320e+18


,dataset_name,split,text,target_label,label_source,is_domain_labeled,speaker_role,conversation_id,turn_id
12396,rusentitweet,val,почему общение со мной такое страггл,negative,existing_sentiment_core,False,NaN,NaN,1.330879e+18
12397,rusentitweet,val,что,neutral,existing_sentiment_core,False,NaN,NaN,1.251774e+18
12398,rusentitweet,val,ну вот как с ними общаться можно,neutral,existing_sentiment_core,False,NaN,NaN,1.213393e+18
12399,rusentitweet,val,надо бы окна закрыть))),neutral,existing_sentiment_core,False,NaN,NaN,1.263766e+18
12400,rusentitweet,val,"Вот-вот. Привет! Что с ""Легендой""?",neutral,existing_sentiment_core,False,NaN,NaN,1.311706e+18


,dataset_name,split,text,target_label,label_source,is_domain_labeled,speaker_role,conversation_id,turn_id
0,rusentiment,test,"Александр, тебе к лицу эта пушка :)",neutral,existing_sentiment_core,False,NaN,NaN,NaN
1,rusentiment,test,"Скоро ты вернешься домой, грязный, не бритый но такой любимый ❤",positive,existing_sentiment_core,False,NaN,NaN,NaN
2,rusentiment,test,помниш...)),neutral,existing_sentiment_core,False,NaN,NaN,NaN
3,rusentiment,test,Наши красавцы. 1:3 в первом периоде и 7:3 в итоге.,positive,existing_sentiment_core,False,NaN,NaN,NaN
4,rusentiment,test,20% усилий приносят 80% результата,neutral,existing_sentiment_core,False,NaN,NaN,NaN


In [460]:
# Summary-таблицы
split_summary_df = (
    final_supervised_df["split"]
    .value_counts(dropna=False)
    .rename_axis("split")
    .reset_index(name="count")
)

label_summary_df = (
    final_supervised_df["target_label"]
    .value_counts(dropna=False)
    .rename_axis("target_label")
    .reset_index(name="count")
)

dataset_summary_df = (
    final_supervised_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

split_label_summary_df = (
    final_supervised_df
    .groupby(["split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)

dataset_split_label_summary_df = (
    final_supervised_df
    .groupby(["dataset_name", "split", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "split", "count"], ascending=[True, True, False])
)

display(split_summary_df)
display(label_summary_df)
display(dataset_summary_df)

,split,count
0,train,26596
1,test,4515
2,val,4120


,target_label,count
0,neutral,18449
1,positive,9486
2,negative,7296


,dataset_name,count
0,rusentiment,23278
1,rusentitweet,11053
2,teacher_student_dialogues_ru,600
3,student_feedback_ru,300


In [461]:
# Metadata
final_dataset_metadata = {
    "dataset_name": "final_supervised_dataset",
    "rows_total": int(len(final_supervised_df)),
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "columns_total": int(len(final_supervised_df.columns)),
    "compact_columns": list(train_compact_df.columns),
    "sources": dataset_summary_df.to_dict(orient="records"),
    "labels": label_summary_df.to_dict(orient="records"),
    "splits": split_summary_df.to_dict(orient="records"),
    "split_label_distribution": split_label_summary_df.to_dict(orient="records"),
    "dataset_split_label_distribution": dataset_split_label_summary_df.to_dict(orient="records"),
    "notes": {
        "sentiment_core": "existing sentiment data with normalized splits",
        "domain_gold": "manually labeled domain batch merged back from sheet",
        "domain_dialogues_split_rule": "group split by conversation_id for teacher_student_dialogues_ru",
        "feedback_split_rule": "row stratified split for student_feedback_ru",
    }
}

## Записываем файлы на диск

In [462]:
# Полные версии
final_supervised_df.to_csv(final_supervised_csv_path, index=False)
final_supervised_df.to_parquet(final_supervised_parquet_path, index=False)

train_df.to_csv(train_full_csv_path, index=False)
train_df.to_parquet(train_full_parquet_path, index=False)

val_df.to_csv(val_full_csv_path, index=False)
val_df.to_parquet(val_full_parquet_path, index=False)

test_df.to_csv(test_full_csv_path, index=False)
test_df.to_parquet(test_full_parquet_path, index=False)

# Компактные версии
train_compact_df.to_csv(train_compact_csv_path, index=False)
train_compact_df.to_parquet(train_compact_parquet_path, index=False)

val_compact_df.to_csv(val_compact_csv_path, index=False)
val_compact_df.to_parquet(val_compact_parquet_path, index=False)

test_compact_df.to_csv(test_compact_csv_path, index=False)
test_compact_df.to_parquet(test_compact_parquet_path, index=False)

# Summary + metadata
split_summary_csv_path = METADATA_DIR / "split_summary.csv"
label_summary_csv_path = METADATA_DIR / "label_summary.csv"
dataset_summary_csv_path = METADATA_DIR / "dataset_summary.csv"
split_label_summary_csv_path = METADATA_DIR / "split_label_summary.csv"
dataset_split_label_summary_csv_path = METADATA_DIR / "dataset_split_label_summary.csv"
metadata_json_path = METADATA_DIR / "final_supervised_dataset_metadata.json"

split_summary_df.to_csv(split_summary_csv_path, index=False)
label_summary_df.to_csv(label_summary_csv_path, index=False)
dataset_summary_df.to_csv(dataset_summary_csv_path, index=False)
split_label_summary_df.to_csv(split_label_summary_csv_path, index=False)
dataset_split_label_summary_df.to_csv(dataset_split_label_summary_csv_path, index=False)

with open(metadata_json_path, "w", encoding="utf-8") as f:
    json.dump(final_dataset_metadata, f, ensure_ascii=False, indent=2)

print("Сохранено:")
print(" -", final_supervised_csv_path)
print(" -", final_supervised_parquet_path)
print(" -", train_full_csv_path)
print(" -", train_full_parquet_path)
print(" -", val_full_csv_path)
print(" -", val_full_parquet_path)
print(" -", test_full_csv_path)
print(" -", test_full_parquet_path)
print(" -", train_compact_csv_path)
print(" -", train_compact_parquet_path)
print(" -", val_compact_csv_path)
print(" -", val_compact_parquet_path)
print(" -", test_compact_csv_path)
print(" -", test_compact_parquet_path)
print(" -", split_summary_csv_path)
print(" -", label_summary_csv_path)
print(" -", dataset_summary_csv_path)
print(" -", split_label_summary_csv_path)
print(" -", dataset_split_label_summary_csv_path)
print(" -", metadata_json_path)

Сохранено:
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/final_supervised_dataset.csv
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/final_supervised_dataset.parquet
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_full.csv
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_full.parquet
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/validation/val_full.csv
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/validation/val_full.parquet
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/test/test_full.csv
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/test/test_full.parquet
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_compact.csv
 - /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_compact.parquet
 - /content/drive/MyDrive/tutors_sentiment_pr

In [463]:
saved_ready_files_df = pd.DataFrame([
    {"artifact": "final_supervised_csv", "path": str(final_supervised_csv_path), "exists": final_supervised_csv_path.exists()},
    {"artifact": "final_supervised_parquet", "path": str(final_supervised_parquet_path), "exists": final_supervised_parquet_path.exists()},
    {"artifact": "train_full_csv", "path": str(train_full_csv_path), "exists": train_full_csv_path.exists()},
    {"artifact": "train_full_parquet", "path": str(train_full_parquet_path), "exists": train_full_parquet_path.exists()},
    {"artifact": "val_full_csv", "path": str(val_full_csv_path), "exists": val_full_csv_path.exists()},
    {"artifact": "val_full_parquet", "path": str(val_full_parquet_path), "exists": val_full_parquet_path.exists()},
    {"artifact": "test_full_csv", "path": str(test_full_csv_path), "exists": test_full_csv_path.exists()},
    {"artifact": "test_full_parquet", "path": str(test_full_parquet_path), "exists": test_full_parquet_path.exists()},
    {"artifact": "train_compact_csv", "path": str(train_compact_csv_path), "exists": train_compact_csv_path.exists()},
    {"artifact": "train_compact_parquet", "path": str(train_compact_parquet_path), "exists": train_compact_parquet_path.exists()},
    {"artifact": "val_compact_csv", "path": str(val_compact_csv_path), "exists": val_compact_csv_path.exists()},
    {"artifact": "val_compact_parquet", "path": str(val_compact_parquet_path), "exists": val_compact_parquet_path.exists()},
    {"artifact": "test_compact_csv", "path": str(test_compact_csv_path), "exists": test_compact_csv_path.exists()},
    {"artifact": "test_compact_parquet", "path": str(test_compact_parquet_path), "exists": test_compact_parquet_path.exists()},
    {"artifact": "split_summary_csv", "path": str(split_summary_csv_path), "exists": split_summary_csv_path.exists()},
    {"artifact": "label_summary_csv", "path": str(label_summary_csv_path), "exists": label_summary_csv_path.exists()},
    {"artifact": "dataset_summary_csv", "path": str(dataset_summary_csv_path), "exists": dataset_summary_csv_path.exists()},
    {"artifact": "split_label_summary_csv", "path": str(split_label_summary_csv_path), "exists": split_label_summary_csv_path.exists()},
    {"artifact": "dataset_split_label_summary_csv", "path": str(dataset_split_label_summary_csv_path), "exists": dataset_split_label_summary_csv_path.exists()},
    {"artifact": "metadata_json", "path": str(metadata_json_path), "exists": metadata_json_path.exists()},
])

display(saved_ready_files_df)

,artifact,path,exists
0,final_supervised_csv,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/final_supervised_dataset.csv,True
1,final_supervised_parquet,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/final_supervised_dataset.parquet,True
2,train_full_csv,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_full.csv,True
3,train_full_parquet,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_full.parquet,True
4,val_full_csv,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/validation/val_full.csv,True
5,val_full_parquet,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/validation/val_full.parquet,True
6,test_full_csv,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/test/test_full.csv,True
7,test_full_parquet,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/test/test_full.parquet,True
8,train_compact_csv,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_compact.csv,True
9,train_compact_parquet,/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready/train/train_compact.parquet,True


## Финальный итог `03_build_target_dataset`

В этом ноутбуке:
- загрузили cleaned-артефакты из `02_data_preparation`;
- выделили базовый sentiment-слой и доменный пул;
- собрали и разметили доменный батч;
- построили `domain_gold_df`;
- объединили доменный gold-слой с sentiment-ядром;
- получили финальный supervised dataset;
- сформировали `train / val / test`;
- сохранили готовые наборы в `04_datasets_ready`.


In [464]:
final_03_summary = {
    "final_supervised_rows": int(len(final_supervised_df)),
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "saved_to": str(DATASETS_READY_DIR),
}

final_03_summary

{'final_supervised_rows': 35231,
 'train_rows': 26596,
 'val_rows': 4120,
 'test_rows': 4515,
 'saved_to': '/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready'}